# Q-M2H: Quantum-Enhanced Multimodal & Missing-Modality-Harmonized Federated Learning
### Heart-disease risk stratification with actionable and adaptive interventions — full implementation

This notebook implements every block of the paper's pipeline end-to-end (PyTorch, runs on Colab CPU or GPU):

| Paper section | Implemented here |
|---|---|
| 3.1 Data | UCI/Kaggle Heart Disease (auto-fetched from your GitHub repo) + **simulated** ECG / echo / angiography / EHR-text / wearable modalities |
| 3.2.1 Numerical | VAE+KNN imputation (Eq.1), Robust Rank-Gaussian (Eq.2), Kalman-smoothed interpolation (Eq.3) |
| 3.2.2 ECG | WPD-AT denoising (Eq.4), MS-RPA R-peaks (Eq.5), synchrosqueezed transform CSST (Eq.6) |
| 3.2.3 Imaging | adaptive bilateral-TV (Eq.7), ROI extraction (Eq.8, classical stand-in for YOLOv9-CardioMask), piecewise intensity normalisation PCH (Eq.9) |
| 3.2.4 Text | tokenizer + clinical-entity/negation extraction (Eq.10-11, rule-based stand-in), hierarchical embeddings HiTraE (Eq.12) |
| 3.2.5 MMA | Dual-Latent Context Vectors (Eq.13) + Mask-informed Quantum Gating (Eq.14) |
| 3.3 Quantum | qutrit embedding QHE (Eq.15), parameterised multi-entanglement layers PMEL (Eq.16), virtual distillation + randomised compiling VDRC (Eq.17), quantum-classical cross attention QCCA (Eq.18) — exact state-vector / density-matrix simulation in PyTorch |
| 3.4 Federated | FedAvg-style aggregation (Eq.19-20), FedYogi (Eq.21), trimmed mean (Eq.22), lattice HE (Eq.23-24, toy RLWE), DP + Renyi accounting (Eq.25-26), top-k + error feedback (Eq.27-28) |
| 3.5 Training | 5 modality experts (Eq.29-33), entropy-aware sparse routing (Eq.34), encrypted aggregation (Eq.35), AGC + Lookahead (Eq.36), ShakeDrop-Lite (Eq.37) |
| 3.7 Actionable | MC-Bayesian calibration (Eq.38), SHAP-QM (Eq.39), gradient counterfactuals (Eq.40), constrained meta-RL recommender (Eq.41) |
| 4 Results | per-modality tables vs ResNet / XGBoost / LSTM / SVM, 5-fold CV, paired t-tests, confusion matrices, robustness to missing modalities |

**Please read before quoting numbers (honest-reporting notes)**
1. The public Heart-Disease dataset is *tabular only*. The paper's ECG / imaging / EHR-text / wearable / angiography modalities do not exist for these patients, so this notebook **simulates them from each patient's clinical row and risk class** (`CFG.modality_noise` controls difficulty). Results on simulated modalities measure the *pipeline*, not clinical performance. If you have real multimodal data, replace `build_modalities()` and keep everything else.
2. The Kaggle `heart.csv` (1025 rows) contains many exact duplicate rows (~300 unique patients). Random splits then leak test patients into training and inflate accuracy toward 99-100%. `CFG.drop_duplicates=True` removes them. Expect lower — but trustworthy — numbers than the paper's ~98%.
3. "Risk class" (low / medium / high) is derived from `target` plus a severity score (oldpeak, ca, age, thalach, ...). It is a documented proxy, not a clinical gold label.
4. The HE module is a **toy RLWE (BFV-style) additively-homomorphic scheme** for demonstration — not production-secure parameters. Use TenSEAL / OpenFHE / Pyfhel for deployment. With only a handful of clients and ~10^5 parameters, *meaningful* client-level differential privacy (small epsilon) is not achievable without destroying accuracy; the code implements the mechanism + Renyi accounting and prints the honest epsilon.

## 0. Setup

In [ ]:
# Colab: torch is preinstalled. PyWavelets (wavelet-packet denoising) and xgboost are optional (graceful fallbacks exist).
import sys, subprocess
for pkg in ['PyWavelets', 'xgboost']:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False, timeout=300)
    except Exception as e:
        print('optional package skipped:', pkg, e)

In [ ]:
import os, math, time, json, copy, random, re, itertools, warnings, urllib.request
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sps
from scipy import ndimage, stats
from scipy.optimize import minimize_scalar
from scipy.linalg import expm
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.impute import KNNImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, matthews_corrcoef
try:
    import pywt
    HAS_PYWT = True
except Exception:
    HAS_PYWT = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGB = False
print('PyWavelets:', HAS_PYWT, '| xgboost:', HAS_XGB, '| torch', torch.__version__)

In [ ]:
class CFG:
    seed = 42
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    out_dir = 'qm2h_outputs'
    # ---------------- data ----------------
    local_csv = None                # e.g. '/content/heart.csv' to force a local file
    github_api = 'https://api.github.com/repos/code-iot278/Heart-disease-classification/contents/'
    invert_target = False           # set True if your CSV codes 1 = healthy
    drop_duplicates = True          # strongly recommended (see note 2)
    inject_missing = 0.08           # MCAR missingness injected into numeric clinical columns
    modality_noise = 1.0            # >1 = harder simulated modalities
    # ---------------- model ----------------
    MODS = ['numerical', 'ecg', 'echo', 'angio', 'text', 'wearable']
    d = 32                          # shared embedding size (must be even)
    nq = 4                          # qutrits per modality circuit (Hilbert dim 3^4 = 81)
    q_layers = 3                    # PMEL depth
    qnoise_p = 0.05                 # simulated depolarising noise
    qnoise_eps = 0.10               # simulated coherent over-rotation noise
    topk_experts = 3
    ent_lambda = 0.5                # entropy penalty in router (Eq.34)
    # ---------------- federated training ----------------
    n_clients = 5
    rounds = 20
    local_epochs = 2
    batch = 16
    lr = 3e-3
    agc = 0.1                       # adaptive gradient clipping factor (Eq.36)
    mod_drop = 0.2                  # training-time modality dropout
    server_lr = 0.03
    yogi_tau = 1e-2
    topk_frac = 0.3                 # gradient sparsification (Eq.27)
    dp_clip = 1.0
    dp_noise = 0.002                # noise multiplier sigma (see note 4)
    dp_delta = 1e-5
    use_he = True                   # encrypted aggregation (Eq.23-24, 35)
    trim = 1                        # trimmed-mean parameter when use_he=False (Eq.22)
    byzantine = None                # e.g. 0 -> client 0 sends sign-flipped updates
    # ---------------- evaluation ----------------
    n_folds = 5
    cv_rounds = 12
    run_cv = True
    baseline_epochs = 60
    mc_samples = 20
    run_ablation = False

random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
os.makedirs(CFG.out_dir, exist_ok=True)
print('device:', CFG.device)

## 1. Data collection (Section 3.1)
The loader tries (1) a local `heart.csv`, (2) the CSV in your GitHub repo, (3) a synthetic Cleveland-like generator (demo only).

In [ ]:
FEATS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
ALIASES = {'thalch': 'thalach', 'num': 'target', 'condition': 'target', 'output': 'target',
           'heartdisease': 'target', 'heart_disease': 'target', 'class': 'target'}

def _standardise(df):
    df = df.copy()
    df.columns = [ALIASES.get(c.strip().lower(), c.strip().lower()) for c in df.columns]
    miss = [c for c in FEATS + ['target'] if c not in df.columns]
    if miss:
        raise ValueError(f'missing columns {miss}')
    return df[FEATS + ['target']].apply(pd.to_numeric, errors='coerce')

def synthetic_cleveland(n=1025, seed=0):
    r = np.random.default_rng(seed); t = r.integers(0, 2, n)
    df = pd.DataFrame({
        'age': np.clip(r.normal(54 + 4 * t, 8), 29, 77).round(),
        'sex': (r.random(n) < 0.55 + 0.2 * t).astype(int),
        'cp': np.where(t == 1, r.choice(4, n, p=[.3, .1, .25, .35]), r.choice(4, n, p=[.45, .25, .25, .05])),
        'trestbps': np.clip(r.normal(128 + 6 * t, 16), 90, 200).round(),
        'chol': np.clip(r.normal(240 + 10 * t, 45), 120, 500).round(),
        'fbs': (r.random(n) < 0.12 + 0.05 * t).astype(int),
        'restecg': r.choice(3, n, p=[.5, .48, .02]),
        'thalach': np.clip(r.normal(160 - 22 * t, 19), 70, 200).round(),
        'exang': (r.random(n) < 0.15 + 0.4 * t).astype(int),
        'oldpeak': np.clip(r.gamma(1.2, 0.6 + 0.7 * t), 0, 6.2).round(1),
        'slope': np.where(t == 1, r.choice(3, n, p=[.1, .6, .3]), r.choice(3, n, p=[.07, .3, .63])),
        'ca': np.where(t == 1, r.choice(4, n, p=[.35, .3, .2, .15]), r.choice(4, n, p=[.8, .12, .06, .02])),
        'thal': np.where(t == 1, r.choice(4, n, p=[.01, .1, .55, .34]), r.choice(4, n, p=[.01, .1, .3, .59])),
        'target': t})
    return df

def _gh_list(url):
    with urllib.request.urlopen(url, timeout=20) as r:
        return json.load(r)

def load_heart():
    for p in [CFG.local_csv, 'heart.csv', '/content/heart.csv', '/content/Heart-disease-classification/heart.csv']:
        if p and os.path.exists(p):
            return _standardise(pd.read_csv(p)), f'local:{p}'
    try:
        items = _gh_list(CFG.github_api)
        for it in [i for i in items if i['type'] == 'dir']:      # one folder level deep
            try:
                items += _gh_list(it['url'])
            except Exception:
                pass
        for it in items:
            if it['type'] == 'file' and it['name'].lower().endswith('.csv'):
                try:
                    return _standardise(pd.read_csv(it['download_url'])), it['download_url']
                except Exception:
                    continue
    except Exception as e:
        print('GitHub fetch failed:', repr(e)[:150])
    print('!! Using SYNTHETIC Cleveland-like data (demo only). Upload heart.csv and set CFG.local_csv for real data.')
    return synthetic_cleveland(), 'synthetic'

df, DATA_SRC = load_heart()
df = df.dropna(subset=['target']).copy()
df['target'] = (df['target'] > 0).astype(int)
if CFG.invert_target:
    df['target'] = 1 - df['target']
for c in ['ca', 'thal']:
    df[c] = df[c].fillna(df[c].mode()[0])
df = df.dropna().reset_index(drop=True)
n_before = len(df)
if CFG.drop_duplicates:
    df = df.drop_duplicates().reset_index(drop=True)
print(f'source: {DATA_SRC}\nrows: {n_before} -> {len(df)} after duplicate removal ({n_before - len(df)} duplicates dropped)')
print(df['target'].value_counts().to_dict(), '(1 = heart disease)')

In [ ]:
# 3-class risk label: low = no disease; medium/high = disease split by a severity score (documented proxy)
def make_risk_classes(d_):
    z = lambda c: (d_[c] - d_[c].mean()) / (d_[c].std() + 1e-9)
    sev = 0.8 * z('oldpeak') + 0.7 * z('ca') + 0.4 * z('age') - 0.5 * z('thalach') + 0.3 * z('exang') + 0.2 * z('chol')
    pos = d_['target'].values == 1
    thr = np.median(sev[pos])
    y = np.zeros(len(d_), int)
    y[pos & (sev <= thr)] = 1
    y[pos & (sev > thr)] = 2
    return y

Y = make_risk_classes(df)
N = len(df); C = 3
RISK_NAMES = ['low', 'medium', 'high']
print('risk-class counts:', dict(zip(RISK_NAMES, np.bincount(Y))))

CONT = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']; ORD = ['ca']; BIN = ['sex', 'fbs', 'exang']; CAT = ['cp', 'restecg', 'slope', 'thal']
RAWC = df[CONT + ORD].values.astype(float)                      # (N,6) clean values (kept for simulation + reference)
rng_m = np.random.default_rng(CFG.seed + 1)
_miss = rng_m.random(RAWC.shape) < CFG.inject_missing
for i in np.where((~_miss).sum(1) < 3)[0]:
    _miss[i, :] = False
RAWC_OBS = RAWC.copy(); RAWC_OBS[_miss] = np.nan                  # what the model is allowed to see
_dum = pd.get_dummies(df[CAT].astype(int).astype(str)).astype(float)
BINCAT = np.concatenate([df[BIN].values.astype(float), _dum.values], 1)
BINCAT_NAMES = BIN + list(_dum.columns)
NUM_NAMES = CONT + ['ca'] + BINCAT_NAMES
DNUM = len(NUM_NAMES)
print('numeric feature dim:', DNUM, '| injected-missing cells:', int(_miss.sum()))

## 2. Simulated multimodal records (replace with real data if available)
Each modality is generated from the patient's clinical row plus a noisy latent copy of the risk class, so modalities are informative but imperfect.

In [ ]:
FS = 128; T_ECG = 384; IMG = 32; T_WEAR = 48
CP_TXT = {0: 'typical angina', 1: 'atypical angina', 2: 'non-anginal chest pain', 3: 'asymptomatic presentation'}
_sig = lambda z: 1 / (1 + np.exp(-z))

def gen_ecg(row, lat, rng, T=T_ECG, fs=FS):
    hr = np.clip(row.thalach * 0.45 + 25 - 4 * lat + rng.normal(0, 4), 50, 120)
    rr = 60.0 / hr; jit = max(0.005, 0.04 - 0.012 * lat)
    t = np.arange(T) / fs; x = np.zeros(T)
    tb = rng.uniform(0.1, 0.4)
    st_dep = 0.06 * float(row.oldpeak) * (0.5 + 0.5 * lat / 2)
    while tb < T / fs + 0.4:
        for c, a, w in [(-0.20, 0.12, 0.025), (-0.04, -0.12, 0.010), (0.0, 1.0, 0.012), (0.04, -0.20, 0.012),
                        (0.14, -st_dep, 0.045), (0.27, 0.30 - 0.14 * lat, 0.050)]:
            x += a * np.exp(-0.5 * ((t - tb - c) / w) ** 2)
        tb += rr * (1 + rng.normal(0, jit))
    nz = CFG.modality_noise
    x += 0.15 * np.sin(2 * np.pi * 0.3 * t + rng.uniform(0, 6.28)) + 0.06 * nz * np.sin(2 * np.pi * 50 * t) + rng.normal(0, 0.06 * nz, T)
    return x.astype(np.float32)

def gen_echo(lat, rng, n=IMG):
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    cy, cx = n / 2 + rng.normal(0, 1.0), n / 2 + rng.normal(0, 1.0)
    ry = 6.0 + 1.3 * lat + rng.normal(0, .6); rx = 4.5 + 0.9 * lat + rng.normal(0, .5)
    wall = max(1.0, 3.8 - 0.8 * lat + rng.normal(0, .4))
    dd = np.sqrt(((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2)
    img = np.full((n, n), 0.12, np.float32)
    img[(dd >= 1) & (dd < 1 + wall / ry)] = 0.85 - 0.1 * lat
    img[dd < 1] = 0.05
    img = ndimage.gaussian_filter(img, 0.8)
    img = img * rng.gamma(10, 1 / 10, size=img.shape) + rng.normal(0, 0.07 * CFG.modality_noise, img.shape)
    return np.clip(img, 0, 1).astype(np.float32)

def gen_angio(lat, ca, rng, n=IMG):
    img = np.full((n, n), 0.78, np.float32); ys = np.arange(n); xx = np.arange(n)[None, :]
    for k in range(3):
        x0 = 6 + k * 10 + rng.normal(0, 1.2); A = rng.uniform(1.5, 3); f = rng.uniform(0.8, 1.6); ph = rng.uniform(0, 6.28)
        cxv = x0 + A * np.sin(2 * np.pi * f * ys / n + ph)
        sten = np.clip(0.18 * lat + 0.22 * (ca > k) + rng.normal(0, .12), 0, 0.92)
        w = 1.8 * (1 - sten * np.exp(-((ys - rng.uniform(8, 24)) / 3.0) ** 2))
        img -= 0.55 * np.exp(-((xx - cxv[:, None]) / w[:, None]) ** 2)
    img = ndimage.gaussian_filter(img, 0.6) + rng.normal(0, 0.05 * CFG.modality_noise, img.shape)
    return np.clip(img, 0, 1).astype(np.float32)

def gen_wear(lat, rng, T=T_WEAR):
    t = np.arange(T) / T
    rest = 62 + 2.5 * lat + rng.normal(0, 3); amp = 9 - 2.2 * lat + rng.normal(0, 1.2); ph = rng.normal(0, 0.3)
    circ = np.sin(2 * np.pi * (t - 0.3 - ph / 6))
    act = np.clip(circ * 0.9 + 0.2, 0, None) * (1.2 - 0.22 * lat) + np.abs(rng.normal(0, .15, T))
    nz = CFG.modality_noise
    hr = rest + amp * circ + 8 * act + rng.normal(0, 2.5 * nz, T)
    steps = np.clip(act * 600 + rng.normal(0, 40 * nz, T), 0, None)
    hrv = 52 - 7 * lat + 6 * np.cos(2 * np.pi * t) + rng.normal(0, 4 * nz, T)
    X = np.stack([hr, steps, hrv], 1)
    X[rng.random(T) < .08] = np.nan                                  # random sensor drops
    if rng.random() < .5:
        s0 = rng.integers(0, T - 6); X[s0:s0 + rng.integers(3, 7)] = np.nan   # burst drop-out
    return X.astype(np.float32)

def gen_note(row, lat, rng):
    s = []
    s.append(('hpi', f"{int(row.age)}-year-old {'woman' if row.sex == 0 else 'man'} with {CP_TXT[int(row.cp) % 4]}."))
    s.append(('hpi', 'Reports exertional dyspnea.' if rng.random() < _sig(1.3 * lat - 0.9) else 'Denies exertional dyspnea.'))
    s.append(('hpi', 'Reports palpitations.' if rng.random() < _sig(0.8 * lat - 1.0) else 'Denies palpitations.'))
    s.append(('hpi', 'Reports syncope.' if rng.random() < _sig(0.6 * lat - 2.0) else 'Denies syncope.'))
    if rng.random() < 0.3:
        s.append(('hpi', 'Family history of premature coronary disease.'))
    s.append(('exam', 'Blood pressure elevated.' if row.trestbps + rng.normal(0, 8) > 135 else 'Blood pressure within normal limits.'))
    s.append(('exam', 'Mild peripheral edema noted.' if rng.random() < _sig(lat - 1.5) else 'No peripheral edema.'))
    if rng.random() < 0.5:
        s.append(('exam', 'Lungs clear to auscultation.'))
    s.append(('studies', 'Resting ECG shows ST-T abnormality.' if rng.random() < _sig(1.2 * lat - 0.8) else 'Resting ECG shows normal sinus rhythm.'))
    s.append(('studies', 'Exercise tolerance reduced.' if rng.random() < _sig(1.1 * lat - 0.7) else 'Exercise tolerance preserved.'))
    s.append(('studies', 'Echocardiogram suggests mildly reduced ejection fraction.' if rng.random() < _sig(1.6 * lat - 1.6) else 'Echocardiogram shows preserved ejection fraction.'))
    s.append(('studies', 'Lipid panel above target.' if row.chol + rng.normal(0, 25) > 240 else 'Lipid panel at goal.'))
    k = int(np.clip(np.round(lat + rng.normal(0, .7)), 0, 2))
    s.append(('assess', f"Impression: {['low', 'intermediate', 'high'][k]} concern for obstructive coronary disease."))
    s.append(('assess', ['Plan: lifestyle counselling.', 'Plan: start statin and schedule stress testing.',
                         'Plan: urgent cardiology referral and angiography.'][int(np.clip(k + rng.integers(-1, 2), 0, 2))]))
    return s

def build_modalities(df_, y, seed=CFG.seed):
    rng = np.random.default_rng(seed + 7); n = len(df_)
    sd = {'ecg': .75, 'echo': .60, 'angio': .55, 'text': .80, 'wear': .80}
    lat = {k: np.clip(y + rng.normal(0, v * CFG.modality_noise, n), -0.5, 2.5) for k, v in sd.items()}
    ecg = np.stack([gen_ecg(df_.iloc[i], lat['ecg'][i], rng) for i in range(n)])
    echo = np.stack([gen_echo(lat['echo'][i], rng) for i in range(n)])
    angio = np.stack([gen_angio(lat['angio'][i], df_.iloc[i].ca, rng) for i in range(n)])
    wear = np.stack([gen_wear(lat['wear'][i], rng) for i in range(n)])
    notes = [gen_note(df_.iloc[i], lat['text'][i], rng) for i in range(n)]
    return dict(ecg=ecg, echo=echo, angio=angio, wear=wear, notes=notes)

RAW_MM = build_modalities(df, Y)
print({k: (v.shape if hasattr(v, 'shape') else len(v)) for k, v in RAW_MM.items()})
print('example note:', ' '.join(s for _, s in RAW_MM['notes'][0]))

## 3. Preprocessing (Section 3.2)

In [ ]:
# ---- 3.2.1(c) Kalman-smoothed interpolation (Eq.3): local-level model, forward filter + RTS backward smoother
def kalman_rts(y, q=0.1, r=0.3):
    n = len(y); x = np.zeros(n); P = np.zeros(n); xp = np.zeros(n); Pp = np.zeros(n)
    if np.all(np.isnan(y)):
        return np.zeros(n)
    xm = np.nanmean(y); Pm = 1.0
    for t in range(n):
        xpred, Ppred = xm, Pm + q
        xp[t], Pp[t] = xpred, Ppred
        if np.isnan(y[t]):
            xm, Pm = xpred, Ppred
        else:
            K = Ppred / (Ppred + r); xm = xpred + K * (y[t] - xpred); Pm = (1 - K) * Ppred
        x[t], P[t] = xm, Pm
    xs = x.copy()
    for t in range(n - 2, -1, -1):
        J = P[t] / Pp[t + 1]; xs[t] = x[t] + J * (xs[t + 1] - xp[t + 1])
    return xs

# ---- 3.2.2(a) Wavelet-packet denoising with band-adaptive soft thresholds (Eq.4)
def wpd_denoise(x, wavelet='db4', level=3, fs=FS):
    x = np.asarray(x, float)
    x = sps.sosfiltfilt(sps.butter(2, 0.5, 'highpass', fs=fs, output='sos'), x)      # baseline wander
    b, a = sps.iirnotch(50.0, 30.0, fs=fs); x = sps.filtfilt(b, a, x)                  # powerline
    if not HAS_PYWT:
        return sps.sosfiltfilt(sps.butter(4, 40, 'lowpass', fs=fs, output='sos'), x)
    wp = pywt.WaveletPacket(x, wavelet, mode='symmetric', maxlevel=level)
    for k, node in enumerate(wp.get_level(level, 'freq')):
        if k == 0:
            continue                                                                    # keep lowest band (P-QRS-T morphology)
        c = node.data
        sigma = np.median(np.abs(c)) / 0.6745 + 1e-9
        node.data = pywt.threshold(c, sigma * np.sqrt(2 * np.log(len(c))) * (1 + 0.15 * k), 'soft')
    return wp.reconstruct(update=False)[:len(x)]

# ---- 3.2.2(b) Multi-scale R-peak aggregation with graph clustering (Eq.5)
def ms_rpa(x, fs=FS, scales=(0.02, 0.04, 0.08), w=(0.3, 0.4, 0.3), tol=0.05, conf=0.25):
    e = np.gradient(x) ** 2; cands = []
    for s, ws in zip(scales, w):
        win = max(3, int(s * fs)); env = np.convolve(e, np.ones(win) / win, 'same'); env = env / (env.max() + 1e-12)
        pk, _ = sps.find_peaks(env, height=0.15, distance=int(0.25 * fs))
        cands += [(p, ws * env[p]) for p in pk]
    if not cands:
        return np.array([], int)
    P = np.array([c[0] for c in cands]); S = np.array([c[1] for c in cands])
    adj = csr_matrix((np.abs(P[:, None] - P[None, :]) <= int(tol * fs)).astype(int))
    nc, lab = connected_components(adj, directed=False)
    peaks = []
    for c in range(nc):
        sel = lab == c
        if S[sel].sum() < conf:
            continue
        pos = int(round(np.average(P[sel], weights=S[sel])))
        lo, hi = max(0, pos - int(0.04 * fs)), min(len(x), pos + int(0.04 * fs) + 1)
        peaks.append(lo + int(np.argmax(x[lo:hi])))
    peaks = np.array(sorted(set(peaks)), int)
    keep = [peaks[0]] if len(peaks) else []
    for p in peaks[1:]:
        if p - keep[-1] >= int(0.25 * fs):
            keep.append(p)
    return np.array(keep, int)

def rpeak_feats(x, peaks, fs=FS):
    if len(peaks) < 3:
        return np.zeros(4)
    rr = np.diff(peaks) / fs
    return np.array([60 / rr.mean(), rr.std() * 1000, np.sqrt(np.mean(np.diff(rr) ** 2)) * 1000, x[peaks].mean()])

# ---- 3.2.2(c) Continuous synchrosqueezed (STFT-based) transform (Eq.6): reassign power to instantaneous frequency
def csst_features(x, fs=FS, L=64, hop=8, nfeat=21):
    n = len(x); pad = L // 2
    xp = np.pad(x, (pad, pad), mode='reflect')
    k = np.arange(-L // 2, L // 2); sg = L / 6.0
    g = np.exp(-0.5 * (k / sg) ** 2); gd = -(k / sg ** 2) * g
    starts = np.arange(0, n, hop)
    fr = np.stack([xp[s:s + L] for s in starts])
    Vg = np.fft.fft(np.fft.ifftshift(fr * g, axes=1), axis=1)[:, :L // 2 + 1]
    Vd = np.fft.fft(np.fft.ifftshift(fr * gd, axes=1), axis=1)[:, :L // 2 + 1]
    om = 2 * np.pi * np.arange(L // 2 + 1) / L
    om_hat = om[None, :] - np.imag(Vd / (Vg + 1e-12))                       # instantaneous frequency (rad/sample)
    bins = np.rint(om_hat * L / (2 * np.pi)).astype(int)
    pw = np.abs(Vg) ** 2
    valid = (bins >= 0) & (bins <= L // 2) & (pw > 1e-4 * pw.max())
    Tmap = np.zeros((len(starts), L // 2 + 1))
    rows = np.repeat(np.arange(len(starts))[:, None], L // 2 + 1, 1)
    np.add.at(Tmap, (rows[valid], bins[valid]), pw[valid])
    return np.log1p(Tmap.sum(0)[:nfeat] / len(starts)), Tmap

# ---- 3.2.3(a) Adaptive bilateral-style TV denoising (Eq.7): edge-adaptive weights on the TV term
def abtv_denoise(imgs, lam=0.10, iters=40, step=0.2, k=0.10, eps=1e-3):
    f = imgs.astype(np.float64); u = f.copy()
    for _ in range(iters):
        gx = np.zeros_like(u); gy = np.zeros_like(u)
        gx[:, :, :-1] = u[:, :, 1:] - u[:, :, :-1]; gy[:, :-1, :] = u[:, 1:, :] - u[:, :-1, :]
        mag = np.sqrt(gx ** 2 + gy ** 2 + eps); wgt = 1.0 / (1.0 + (mag / k) ** 2)
        px, py = wgt * gx / mag, wgt * gy / mag
        div = np.zeros_like(u)
        div[:, :, 0] += px[:, :, 0]; div[:, :, 1:] += px[:, :, 1:] - px[:, :, :-1]
        div[:, 0, :] += py[:, 0, :]; div[:, 1:, :] += py[:, 1:, :] - py[:, :-1, :]
        u = u - step * ((u - f) - lam * div)
    return u.astype(np.float32)

# ---- 3.2.3(b) ROI extraction (Eq.8). Classical stand-in: Otsu + largest component + bbox crop.
#      Plug a trained YOLOv9/seg model here for the paper's YOLOv9-CardioMask.
def otsu(img, bins=64):
    h, edges = np.histogram(img.ravel(), bins=bins, range=(0, 1)); p = h / max(h.sum(), 1)
    w = np.cumsum(p); mu = np.cumsum(p * (edges[:-1] + edges[1:]) / 2); mt = mu[-1]
    sb = (mt * w - mu) ** 2 / (w * (1 - w) + 1e-12)
    i = int(np.argmax(sb)); return (edges[i] + edges[i + 1]) / 2

def resize_bilinear(a, out):
    ys = np.linspace(0, a.shape[0] - 1, out); xs = np.linspace(0, a.shape[1] - 1, out)
    Y, X = np.meshgrid(ys, xs, indexing='ij')
    return ndimage.map_coordinates(a, [Y, X], order=1)

def extract_roi(img, dark=False, out=24):
    t = otsu(img); mask = (img < t) if dark else (img > t)
    mask = ndimage.binary_opening(mask, iterations=1)
    lab, n = ndimage.label(mask); H, W = img.shape
    if n == 0:
        return resize_bilinear(img, out), mask
    k = int(np.argmax(ndimage.sum(mask, lab, range(1, n + 1)))) + 1
    mask = lab == k; ys, xs = np.where(mask)
    y0, y1 = max(ys.min() - 2, 0), min(ys.max() + 3, H); x0, x1 = max(xs.min() - 2, 0), min(xs.max() + 3, W)
    return resize_bilinear(img[y0:y1, x0:x1], out), mask

# ---- 3.2.3(c) Piecewise contrast harmonisation (Eq.9): percentile knots -> fixed targets
def pch_norm(img, p=(0, 25, 50, 75, 100), tgt=(0, .25, .5, .75, 1)):
    src = np.percentile(img, p) + np.arange(len(p)) * 1e-6
    return np.interp(img, src, tgt).astype(np.float32)

# ---- 3.2.4 Text: tokenizer, negation-aware entity extraction, section structure (Eq.10-12 inputs)
ENT_LEX = {'dyspnea', 'palpitations', 'syncope', 'edema', 'angina', 'hypertension', 'st-t', 'ejection', 'statin', 'angiography', 'cardiology', 'lipid'}
NEG_CUES = {'denies', 'no', 'without', 'negative'}
def tokenize(s):
    return re.findall(r"[a-z][a-z\-]+|\d+(?:\.\d+)?", s.lower())
def sentence_to_doc(s):
    toks = tokenize(s); ents = []
    for i, t in enumerate(toks):
        if t in ENT_LEX:
            ents.append(('ent_neg_' if any(c in NEG_CUES for c in toks[max(0, i - 4):i]) else 'ent_') + t)
    return ' '.join(toks + ents)

SEC_ID = {'hpi': 0, 'exam': 1, 'studies': 2, 'assess': 3}
S_MAX = 14; TXT_DIM = 24
SENT_DOCS = [[sentence_to_doc(s) for _, s in note] for note in RAW_MM['notes']]
SENT_SEC = [[SEC_ID[sec] for sec, _ in note] for note in RAW_MM['notes']]
print('text preprocessing ready;', 'sample doc:', SENT_DOCS[0][1])

### Run the per-patient preprocessing (ECG, imaging, wearable) and visualise it

In [ ]:
t0 = time.time()
# ECG
_den = []; _rf = []; _cs = []
for i in range(N):
    xd = wpd_denoise(RAW_MM['ecg'][i]); pk = ms_rpa(xd)
    _den.append(xd); _rf.append(rpeak_feats(xd, pk)); _cs.append(csst_features(xd)[0])
_den = np.stack(_den); _feat = np.concatenate([np.stack(_cs), np.stack(_rf)], 1)
ECG_SIG = (_den / _den.std()).astype(np.float32)
ECG_FEAT = ((_feat - _feat.mean(0)) / (_feat.std(0) + 1e-6)).astype(np.float32)

# Echo + angiography: ABTV -> ROI -> PCH
def process_images(raw, dark):
    den = abtv_denoise(raw)
    rois = []; masks = []
    for i in range(len(den)):
        roi, m = extract_roi(den[i], dark=dark); rois.append(pch_norm(roi)); masks.append(m)
    return den, np.stack(masks), np.stack(rois)[:, None].astype(np.float32)
ECHO_DEN, ECHO_MASK, ECHO = process_images(RAW_MM['echo'], dark=False)
ANGIO_DEN, ANGIO_MASK, ANGIO = process_images(RAW_MM['angio'], dark=True)

# Wearable: Kalman-smoothed interpolation per channel on standardised signals
_w = RAW_MM['wear'].astype(np.float64)
_mu = np.nanmean(_w, (0, 1)); _sd = np.nanstd(_w, (0, 1))
WEAR = np.stack([np.stack([kalman_rts((_w[i, :, c] - _mu[c]) / _sd[c]) for c in range(3)], 1) for i in range(N)]).astype(np.float32)
print(f'preprocessing done in {time.time() - t0:.1f}s | ECG {ECG_SIG.shape}, feat {ECG_FEAT.shape}, echo {ECHO.shape}, angio {ANGIO.shape}, wear {WEAR.shape}')

In [ ]:
i0 = int(np.where(Y == 2)[0][0])
fig, ax = plt.subplots(3, 5, figsize=(15, 8))
for r_, (raw, den, mk, roi, nm) in enumerate([(RAW_MM['echo'], ECHO_DEN, ECHO_MASK, ECHO, 'Echo'), (RAW_MM['angio'], ANGIO_DEN, ANGIO_MASK, ANGIO, 'Angio')]):
    for c_, (im, tt) in enumerate([(raw[i0], 'input'), (den[i0], 'ABTV denoised'), (mk[i0], 'ROI mask'), (roi[i0, 0], 'ROI 24x24'), (roi[i0, 0], 'PCH (final)')]):
        ax[r_, c_].imshow(im, cmap='gray'); ax[r_, c_].set_title(f'{nm}: {tt}', fontsize=9); ax[r_, c_].axis('off')
t_ = np.arange(T_ECG) / FS; pk = ms_rpa(_den[i0])
ax[2, 0].plot(t_, RAW_MM['ecg'][i0], lw=.8); ax[2, 0].set_title('ECG raw', fontsize=9)
ax[2, 1].plot(t_, _den[i0], lw=.8); ax[2, 1].plot(t_[pk], _den[i0][pk], 'rv'); ax[2, 1].set_title('WPD-AT + MS-RPA peaks', fontsize=9)
_, Tm = csst_features(_den[i0])
ax[2, 2].imshow(Tm.T, aspect='auto', origin='lower', extent=[0, T_ECG / FS, 0, FS / 2], cmap='magma'); ax[2, 2].set_title('CSST map (Hz)', fontsize=9)
ax[2, 3].plot(RAW_MM['wear'][i0, :, 0], 'o', ms=3, label='raw HR'); ax[2, 3].plot(WEAR[i0, :, 0] * _sd[0] + _mu[0], label='Kalman'); ax[2, 3].legend(fontsize=7); ax[2, 3].set_title('Wearable HR', fontsize=9)
ax[2, 4].axis('off'); ax[2, 4].text(0, 0.5, 'Note text:\n' + ' '.join(s for _, s in RAW_MM['notes'][i0])[:230] + '...', fontsize=6, wrap=True, va='center')
plt.tight_layout(); plt.savefig(f'{CFG.out_dir}/fig_preprocessing.png', dpi=130); plt.show()

### 3.2.1 Numerical preprocessing: hybrid VAE+KNN imputation (Eq.1) and Robust Rank-Gaussian transform (Eq.2)

In [ ]:
class VAE(nn.Module):
    def __init__(s, p, hid=16, lat=3):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(2 * p, hid), nn.ReLU()); s.mu = nn.Linear(hid, lat); s.lv = nn.Linear(hid, lat)
        s.dec = nn.Sequential(nn.Linear(lat, hid), nn.ReLU(), nn.Linear(hid, p))
    def forward(s, x):
        h = s.enc(x); mu, lv = s.mu(h), s.lv(h).clamp(-6, 4)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * lv)
        return s.dec(z), mu, lv

class HybridImputer:
    # X_hat = M*X + (1-M) * (alpha*VAE_decoder + (1-alpha)*KNN)     (Eq.1)
    def __init__(s, alpha=0.5, k=5, epochs=300):
        s.alpha, s.k, s.epochs = alpha, k, epochs
    def fit(s, X):
        s.mu_, s.sd_ = np.nanmean(X, 0), np.nanstd(X, 0) + 1e-6
        s.knn = KNNImputer(n_neighbors=s.k).fit(X)
        Z = (X - s.mu_) / s.sd_; M = torch.tensor((~np.isnan(Z)).astype(np.float32)); Z0 = torch.tensor(np.nan_to_num(Z).astype(np.float32))
        s.vae = VAE(X.shape[1]); opt = torch.optim.Adam(s.vae.parameters(), lr=5e-3)
        for _ in range(s.epochs):
            Min = M * (torch.rand_like(M) > 0.15).float()                       # extra masking -> denoising VAE
            rec, mu, lv = s.vae(torch.cat([Z0 * Min, Min], 1))
            loss = (((rec - Z0) ** 2) * M).sum() / M.sum() + 1e-3 * (-0.5 * (1 + lv - mu ** 2 - lv.exp()).sum(1).mean())
            opt.zero_grad(); loss.backward(); opt.step()
        return s
    @torch.no_grad()
    def transform(s, X):
        Z = (X - s.mu_) / s.sd_; M = torch.tensor((~np.isnan(Z)).astype(np.float32)); Z0 = torch.tensor(np.nan_to_num(Z).astype(np.float32))
        h = s.vae.enc(torch.cat([Z0 * M, M], 1)); xv = s.vae.dec(s.vae.mu(h)).numpy() * s.sd_ + s.mu_
        xh = s.alpha * xv + (1 - s.alpha) * s.knn.transform(X)
        return np.where(M.numpy() > 0, np.nan_to_num(X), xh)

class RRG:
    # Robust Rank-Gaussian: z = Phi^-1((rank+0.5)/(n+1)), winsorised at +-3 (Eq.2); fitted on the training split only
    def fit(s, x):
        s.s = np.sort(x[~np.isnan(x)]); return s
    def transform(s, x):
        r = (np.searchsorted(s.s, x, 'left') + np.searchsorted(s.s, x, 'right')) / 2
        u = np.clip((r + 0.5) / (len(s.s) + 1), 1e-4, 1 - 1e-4)
        return np.clip(stats.norm.ppf(u), -3, 3).astype(np.float32)
    def inverse(s, z):
        return np.quantile(s.s, np.clip(stats.norm.cdf(z), 0, 1))

# quick sanity check of both
_imp = HybridImputer(epochs=150).fit(RAWC_OBS)
_Xi = _imp.transform(RAWC_OBS)
_mm = _miss
print('imputation RMSE (z-units) on injected-missing cells: VAE+KNN = %.3f | mean-fill = %.3f' % (
    np.sqrt((((_Xi - RAWC) / _imp.sd_)[_mm] ** 2).mean()),
    np.sqrt((((np.nanmean(RAWC_OBS, 0) - RAWC) / _imp.sd_)[_mm] ** 2).mean())))

## 4. Federated split, modality masks and fold-specific features

In [ ]:
BASE_MISS = np.array([0.03, 0.15, 0.25, 0.40, 0.20, 0.25])      # natural per-modality missing rates

def dirichlet_split(train_idx, y, K, alpha=1.0, seed=0, min_size=12):
    rng = np.random.default_rng(seed)
    for attempt in range(300):
        parts = [[] for _ in range(K)]
        for c in range(C):
            ids = train_idx[y[train_idx] == c].copy(); rng.shuffle(ids)
            p = rng.dirichlet(alpha * np.ones(K)); cuts = (np.cumsum(p) * len(ids)).astype(int)[:-1]
            for k, chunk in enumerate(np.split(ids, cuts)):
                parts[k] += chunk.tolist()
        if min(len(p) for p in parts) >= (min_size if attempt < 250 else 4):
            return [np.array(p) for p in parts]
    raise RuntimeError('could not build client split')

def build_masks(n, client_of, K, seed=0):
    # per-client heterogeneity: some hospitals rarely have angiography / ECG etc.
    rng = np.random.default_rng(seed); scale = np.exp(rng.normal(0, 0.4, (K, 6))); scale[:, 0] = 1
    rate = np.where(client_of[:, None] >= 0, np.clip(BASE_MISS[None] * scale[np.clip(client_of, 0, None)], 0, .85), BASE_MISS[None])
    mask = (rng.random((n, 6)) > rate).astype(np.float32)
    mask[mask.sum(1) == 0, 0] = 1
    return mask

NUM_GROUPS = {c: [i] for i, c in enumerate(CONT + ['ca'] + BIN)}
for pref in CAT:
    NUM_GROUPS[pref] = [i for i, nme in enumerate(NUM_NAMES) if nme.startswith(pref + '_')]

def make_features(train_idx):
    # everything that must be *fitted* (imputer, RRG, TF-IDF/SVD) is fitted on train_idx only -> no leakage
    imp = HybridImputer().fit(RAWC_OBS[train_idx]); Xi = imp.transform(RAWC_OBS)
    rrgs = [RRG().fit(Xi[train_idx, j]) for j in range(5)]
    cz = np.stack([rrgs[j].transform(Xi[:, j]) for j in range(5)], 1)
    num = np.concatenate([cz, np.clip(Xi[:, 5:6], 0, 4) / 4.0, BINCAT], 1).astype(np.float32)
    tr_s = [s_ for i in train_idx for s_ in SENT_DOCS[i]]
    tf = TfidfVectorizer(min_df=2, sublinear_tf=True).fit(tr_s)
    svd = TruncatedSVD(TXT_DIM, random_state=0).fit(tf.transform(tr_s))
    T = np.zeros((N, S_MAX, TXT_DIM), np.float32); TM = np.zeros((N, S_MAX), np.float32); TS = np.zeros((N, S_MAX), np.int64)
    for i in range(N):
        docs = SENT_DOCS[i][:S_MAX]; E = svd.transform(tf.transform(docs)); E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-6)
        T[i, :len(docs)] = E; TM[i, :len(docs)] = 1; TS[i, :len(docs)] = SENT_SEC[i][:S_MAX]
    D = dict(num=num, ecg_sig=ECG_SIG, ecg_feat=ECG_FEAT, echo=ECHO, angio=ANGIO, text=T, text_mask=TM, text_sec=TS, wear=WEAR, y=Y)
    D = {k: torch.tensor(v) for k, v in D.items()}
    D['text_sec'] = D['text_sec'].long(); D['y'] = D['y'].long()
    return D, dict(imputer=imp, rrgs=rrgs, tfidf=tf, svd=svd)

## 5. Quantum layers (Section 3.3) — exact simulation of qutrit registers in PyTorch
* **QHE (Eq.15)** each qutrit starts as a normalised 3-level amplitude/phase state computed from the classical embedding; the register is their tensor product.
* **PMEL (Eq.16)** per layer: a local SU(3) rotation `exp(i * sum_k theta_k * lambda_k)` (Gell-Mann basis) on each qutrit, then a ring of controlled-phase entanglers `exp(i * 2pi/3 * g * a * b)` with *learnable strength* `g`.
* **VDRC (Eq.17)** a fixed *coherent* error unitary is **twirled** with random Weyl–Heisenberg operators (randomised compiling turns it into stochastic noise), a depolarising channel is added, and expectations are read out with **virtual distillation** (`Tr(rho^2 O)/Tr(rho^2)`).

In [ ]:
def gell_mann():
    s3 = 1 / np.sqrt(3)
    G = [[[0, 1, 0], [1, 0, 0], [0, 0, 0]], [[0, -1j, 0], [1j, 0, 0], [0, 0, 0]], [[1, 0, 0], [0, -1, 0], [0, 0, 0]],
         [[0, 0, 1], [0, 0, 0], [1, 0, 0]], [[0, 0, -1j], [0, 0, 0], [1j, 0, 0]],
         [[0, 0, 0], [0, 0, 1], [0, 1, 0]], [[0, 0, 0], [0, 0, -1j], [0, 1j, 0]],
         [[s3, 0, 0], [0, s3, 0], [0, 0, -2 * s3]]]
    return torch.tensor(np.array(G), dtype=torch.complex64)

def weyl_op(a, b):
    X = np.roll(np.eye(3), 1, axis=0); Z = np.diag(np.exp(2j * np.pi * np.arange(3) / 3))
    ops = [np.linalg.matrix_power(X, ai) @ np.linalg.matrix_power(Z, bi) for ai, bi in zip(a, b)]
    W = ops[0]
    for o in ops[1:]:
        W = np.kron(W, o)
    return W

class QutritCircuit(nn.Module):
    def __init__(s, d, nq=4, L=3, p=0.05, eps=0.10, K=2, seed=0):
        super().__init__()
        s.nq, s.L, s.dim, s.p, s.K = nq, L, 3 ** nq, p, K
        s.enc = nn.Linear(d, nq * 6)                                           # classical -> amplitudes & phases
        s.theta = nn.Parameter(torch.randn(L, nq, 8) * 0.3)
        s.g = nn.Parameter(torch.rand(L, nq) * 0.5)
        s.register_buffer('GM', gell_mann())
        dig = np.indices((3,) * nq).reshape(nq, -1).astype(np.float32)
        s.register_buffer('pairprod', torch.tensor(np.stack([dig[i] * dig[(i + 1) % nq] for i in range(nq)])))
        s.register_buffer('obs', torch.tensor(np.stack([(dig[i] == a).astype(np.float32) for i in range(nq) for a in range(3)])))
        rng = np.random.default_rng(seed)
        A = rng.normal(size=(s.dim, s.dim)) + 1j * rng.normal(size=(s.dim, s.dim)); H = (A + A.conj().T) / 2
        H = H / np.linalg.norm(H, 2); E = expm(1j * eps * H)                    # small coherent over-rotation
        Ut = [weyl_op(rng.integers(0, 3, nq), rng.integers(0, 3, nq)) for _ in range(K)]
        s.register_buffer('Utw', torch.tensor(np.stack([W @ E @ W.conj().T for W in Ut]), dtype=torch.complex64))   # twirled error
    def _state(s, c):
        B = c.size(0); z = s.enc(c).view(B, s.nq, 2, 3)
        q = torch.polar(F.normalize(z[:, :, 0], dim=-1), z[:, :, 1] * math.pi)  # (B,nq,3) complex single-qutrit states
        psi = q[:, 0]
        for i in range(1, s.nq):
            psi = torch.einsum('bi,bj->bij', psi, q[:, i]).reshape(B, -1)
        for l in range(s.L):
            U = torch.matrix_exp(1j * torch.einsum('qk,kab->qab', s.theta[l].to(torch.complex64), s.GM))   # (nq,3,3)
            psi = psi.reshape(B, *([3] * s.nq))
            for i in range(s.nq):
                psi = torch.movedim(torch.movedim(psi, i + 1, -1) @ U[i].transpose(0, 1), -1, i + 1)
            psi = psi.reshape(B, -1)
            phase = (s.g[l].unsqueeze(-1) * s.pairprod).sum(0) * (2 * math.pi / 3)
            psi = psi * torch.exp(1j * phase)
        return psi
    def forward(s, c, mode='vd'):
        psi = s._state(c)
        if mode == 'ideal':
            return (psi.abs() ** 2) @ s.obs.T
        rho = psi.unsqueeze(-1) * psi.conj().unsqueeze(-2); acc = 0
        for k in range(s.K):
            Uk = s.Utw[k]; acc = acc + Uk @ rho @ Uk.conj().transpose(0, 1)
        rho = (1 - s.p) * acc / s.K + s.p * torch.eye(s.dim, device=c.device) / s.dim
        if mode == 'noisy':
            return torch.diagonal(rho, dim1=-2, dim2=-1).real @ s.obs.T
        d2 = (rho.abs() ** 2).sum(-1)                                          # diag(rho^2) for Hermitian rho
        return (d2 @ s.obs.T) / d2.sum(-1, keepdim=True)                       # virtual distillation, M = 2

# unit tests: unitarity (norm preserved) and that noise/mitigation behave as expected
_qc = QutritCircuit(CFG.d, CFG.nq, CFG.q_layers, CFG.qnoise_p, CFG.qnoise_eps)
_c = torch.randn(64, CFG.d)
with torch.no_grad():
    _ps = _qc._state(_c); _id = _qc(_c, 'ideal'); _no = _qc(_c, 'noisy'); _vd = _qc(_c, 'vd')
print('state norm (should be 1): %.5f' % _ps.abs().pow(2).sum(1).mean().item())
print('mean |error| vs ideal  ->  noisy: %.4f | virtual-distilled: %.4f' % ((_no - _id).abs().mean().item(), (_vd - _id).abs().mean().item()))

## 6. Modality experts, MMA, QCCA, entropy-gated mixture of experts (Sections 3.2.5, 3.3.4, 3.5)

In [ ]:
class ResBlock(nn.Module):
    # ShakeDrop-Lite (Eq.37): y = x + (b + alpha - b*alpha) F(x); b~Bernoulli(p_keep), alpha~U(-1,1); eval uses E[.] = p_keep
    def __init__(s, d, p_keep=0.85, drop=0.1):
        super().__init__()
        s.f = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * d, d)); s.p = p_keep
    def forward(s, x):
        r = s.f(x)
        if s.training:
            b = (torch.rand(x.size(0), 1, device=x.device) < s.p).float()
            a = torch.empty(x.size(0), 1, device=x.device).uniform_(-1, 1)
            return x + (b + a - b * a) * r
        return x + s.p * r

class NumExpert(nn.Module):                      # Eq.29: feature-masked transformer attention over tabular features
    def __init__(s, D, d, tok=16, heads=2):
        super().__init__()
        s.w = nn.Parameter(torch.randn(D, tok) * 0.3); s.b = nn.Parameter(torch.zeros(D, tok)); s.mask = nn.Linear(D, D)
        s.att = nn.TransformerEncoderLayer(tok, heads, 2 * tok, 0.1, batch_first=True, norm_first=True)
        s.out = nn.Sequential(nn.LayerNorm(tok), nn.Linear(tok, d))
    def forward(s, x):
        m = torch.sigmoid(s.mask(x)); h = s.att((x * m).unsqueeze(-1) * s.w + s.b)
        return s.out((h * m.unsqueeze(-1)).sum(1) / (m.sum(1, keepdim=True) + 1e-6))

class ECGExpert(nn.Module):                      # Eq.30: lightweight conv + spectral (CSST) attention + temporal GRU
    def __init__(s, nfeat, d, ch=16):
        super().__init__()
        s.conv = nn.Sequential(nn.Conv1d(1, ch, 7, 2, 3), nn.GELU(), nn.Conv1d(ch, 2 * ch, 5, 2, 2), nn.GELU(), nn.Conv1d(2 * ch, 2 * ch, 5, 2, 2), nn.GELU())
        s.spec = nn.Sequential(nn.Linear(nfeat, 2 * ch), nn.Sigmoid())
        s.gru = nn.GRU(2 * ch, 2 * ch, batch_first=True, bidirectional=True)
        s.out = nn.Sequential(nn.LayerNorm(4 * ch + nfeat), nn.Linear(4 * ch + nfeat, d))
    def forward(s, sig, feat):
        x = s.conv(sig.unsqueeze(1)); x = (x * (1 + s.spec(feat).unsqueeze(-1))).transpose(1, 2)
        o, _ = s.gru(x)
        return s.out(torch.cat([o.mean(1), feat], 1))

class ImgExpert(nn.Module):                      # Eq.31: patch embedding + deformable sampling + transformer attention (Swin-lite)
    def __init__(s, d, ch=24, img=24, patch=4):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(1, ch, patch, patch), nn.GELU())
        s.off = nn.Conv2d(ch, 2, 3, padding=1); nn.init.zeros_(s.off.weight); nn.init.zeros_(s.off.bias)
        s.pos = nn.Parameter(torch.zeros(1, (img // patch) ** 2, ch))
        s.att = nn.TransformerEncoderLayer(ch, 4, 2 * ch, 0.1, batch_first=True, norm_first=True)
        s.out = nn.Sequential(nn.LayerNorm(ch), nn.Linear(ch, d))
    def forward(s, x):
        f = s.stem(x); B, Cc, H, W = f.shape
        off = torch.tanh(s.off(f)) * 0.5
        ys, xs = torch.meshgrid(torch.linspace(-1, 1, H, device=x.device), torch.linspace(-1, 1, W, device=x.device), indexing='ij')
        grid = torch.stack([xs, ys], -1).unsqueeze(0) + off.permute(0, 2, 3, 1)
        f = F.grid_sample(f, grid, align_corners=True, padding_mode='border') + f
        return s.out(s.att(f.flatten(2).transpose(1, 2) + s.pos).mean(1))

class TextExpert(nn.Module):                     # Eq.32: sentence transformer + discourse-graph propagation + section->document pooling
    def __init__(s, din, d, nsec=4, heads=2):
        super().__init__()
        s.nsec = nsec; s.proj = nn.Linear(din, d); s.sec = nn.Embedding(nsec, d)
        s.sent = nn.TransformerEncoderLayer(d, heads, 2 * d, 0.1, batch_first=True, norm_first=True)
        s.gc = nn.Linear(d, d); s.satt = nn.Linear(d, 1); s.out = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d))
    def forward(s, x, m, sec):
        mk = m.clone(); mk[:, 0] = 1                                             # avoid all-masked rows (missing note); MMA overrides them
        h = s.sent(s.proj(x) + s.sec(sec), src_key_padding_mask=(mk == 0))
        S = x.size(1)
        A = (torch.eye(S, device=x.device) + torch.diag(torch.ones(S - 1, device=x.device), 1) + torch.diag(torch.ones(S - 1, device=x.device), -1)).unsqueeze(0)
        A = A * (mk[:, :, None] * mk[:, None, :]); A = A / (A.sum(-1, keepdim=True) + 1e-6)
        h = h + F.gelu(s.gc(A @ h))
        oh = F.one_hot(sec, s.nsec).float() * mk.unsqueeze(-1)
        secv = (oh.transpose(1, 2) @ h) / (oh.sum(1).unsqueeze(-1) + 1e-6)
        a = s.satt(secv).squeeze(-1).masked_fill(oh.sum(1) == 0, -1e9).softmax(-1)
        return s.out((a.unsqueeze(-1) * secv).sum(1))

class WearExpert(nn.Module):                     # Eq.33: GRU with circadian gating
    def __init__(s, din, d, h=32, T=T_WEAR):
        super().__init__()
        s.inp = nn.Linear(din, h); s.cell = nn.GRUCell(h, h); s.circ = nn.Linear(2, h); s.out = nn.Sequential(nn.LayerNorm(h), nn.Linear(h, d))
        ang = 2 * math.pi * torch.arange(T) / T; s.register_buffer('cs', torch.stack([ang.sin(), ang.cos()], -1))
    def forward(s, x):
        B, T, _ = x.shape; u = s.inp(x); h = x.new_zeros(B, s.cell.hidden_size); outs = []
        for t in range(T):
            c = torch.sigmoid(s.circ(s.cs[t])); h = c * s.cell(u[:, t], h) + (1 - c) * h
            outs.append(h)
        return s.out(torch.stack(outs, 1).mean(1))

class MMA(nn.Module):
    # Dual-Latent Context Vectors (Eq.13) + Mask-informed Quantum Gating (Eq.14)
    def __init__(s, M, d):
        super().__init__()
        s.M = M; s.miss = nn.Parameter(torch.zeros(M, d)); s.u0 = nn.Parameter(torch.zeros(M, d))
        s.Wp = nn.ModuleList([nn.Linear(d, d) for _ in range(M)]); s.Wu = nn.ModuleList([nn.Linear(d, d) for _ in range(M)])
        s.fuse = nn.ModuleList([nn.Linear(2 * d, d) for _ in range(M)]); s.th = nn.ModuleList([nn.Linear(d, d // 2) for _ in range(M)])
    def forward(s, h, mask):
        outs = []
        for m in range(s.M):
            hm, mk = h[:, m], mask[:, m:m + 1]
            p = mk * torch.sigmoid(s.Wp[m](hm)) * hm + (1 - mk) * s.miss[m]                      # presence-aware vector
            u = mk * torch.sigmoid(s.Wu[m](hm)) + (1 - mk) * torch.sigmoid(s.u0[m])               # uncertainty-aware vector
            c = s.fuse[m](torch.cat([p, u], -1))
            th = mk * torch.tanh(s.th[m](c)) * math.pi                                            # rotation only if modality present
            a, b = c[:, 0::2], c[:, 1::2]
            c = torch.stack([a * torch.cos(th) - b * torch.sin(th), a * torch.sin(th) + b * torch.cos(th)], -1).flatten(1)
            outs.append(c)
        return torch.stack(outs, 1)

class QCCA(nn.Module):
    # Eq.18: classical queries attend to quantum keys/values (measured expectations are real, so K^dagger = K^T)
    def __init__(s, d, nobs):
        super().__init__()
        s.q = nn.Linear(d, d); s.k = nn.Linear(nobs, d); s.v = nn.Linear(nobs, d); s.o = nn.Linear(d, d); s.ln = nn.LayerNorm(d)
    def forward(s, hc, qf, mask):
        att = (s.q(hc) @ s.k(qf).transpose(1, 2)) / math.sqrt(hc.size(-1))
        att = att.masked_fill(mask[:, None, :] == 0, -1e9).softmax(-1); s.last_att = att.detach()
        return s.ln(hc + s.o(att @ s.v(qf)))

class QM2H(nn.Module):
    def __init__(s, Dnum, n_ecg, d_txt, C=3, d=CFG.d, nq=CFG.nq, L=CFG.q_layers, topk=CFG.topk_experts, lam=CFG.ent_lambda,
                 p=CFG.qnoise_p, eps=CFG.qnoise_eps, use_quantum=True, use_mma=True, use_gate=True):
        super().__init__()
        s.M, s.C, s.topk, s.lam, s.uq, s.umma, s.ug = 6, C, topk, lam, use_quantum, use_mma, use_gate
        s.num = NumExpert(Dnum, d); s.ecg = ECGExpert(n_ecg, d); s.echo = ImgExpert(d); s.angio = ImgExpert(d)
        s.text = TextExpert(d_txt, d); s.wear = WearExpert(3, d)
        s.mma = MMA(6, d)
        if use_quantum:
            s.circ = nn.ModuleList([QutritCircuit(d, nq, L, p, eps, seed=i) for i in range(6)])
        else:
            s.circ = nn.ModuleList([nn.Sequential(nn.Linear(d, nq * 3), nn.Sigmoid()) for _ in range(6)])
        s.qcca = QCCA(d, nq * 3)
        s.blocks = nn.ModuleList([nn.Sequential(ResBlock(d), ResBlock(d)) for _ in range(6)])
        s.heads = nn.ModuleList([nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.2), nn.Linear(d, C)) for _ in range(6)])
        s.router = nn.Linear(d, 1); s.mbias = nn.Parameter(torch.zeros(6))
    def forward(s, b, mask):
        h = torch.stack([s.num(b['num']), s.ecg(b['ecg_sig'], b['ecg_feat']), s.echo(b['echo']), s.angio(b['angio']),
                         s.text(b['text'], b['text_mask'], b['text_sec']), s.wear(b['wear'])], 1)
        hc = s.mma(h, mask) if s.umma else h * mask.unsqueeze(-1)
        qf = torch.stack([s.circ[m](hc[:, m]) for m in range(6)], 1)
        z = s.qcca(hc, qf, mask)
        z = torch.stack([s.blocks[m](z[:, m]) for m in range(6)], 1)
        el = torch.stack([s.heads[m](z[:, m]) for m in range(6)], 1)               # per-expert logits (B,6,C)
        pr = el.softmax(-1); H = -(pr * torch.log(pr + 1e-9)).sum(-1) / math.log(s.C)
        if s.ug:                                                                    # entropy-aware sparse routing (Eq.34)
            score = (s.router(z).squeeze(-1) + s.mbias - s.lam * H).masked_fill(mask == 0, -1e9)
            thr = score.topk(min(s.topk, 6), dim=1).values[:, -1:]
            sel = (score >= thr) & (mask > 0)
            g = score.masked_fill(~sel, -1e9).softmax(-1)
        else:
            g = (mask / mask.sum(1, keepdim=True).clamp_min(1))
        return dict(logits=(g.unsqueeze(-1) * el).sum(1), el=el, gate=g)

def build_model(D, **kw):
    return QM2H(D['num'].shape[1], D['ecg_feat'].shape[1], D['text'].shape[2], **kw)

_D0 = {k: v[:8] for k, v in {'num': torch.randn(N, DNUM), 'ecg_sig': torch.tensor(ECG_SIG), 'ecg_feat': torch.tensor(ECG_FEAT), 'echo': torch.tensor(ECHO), 'angio': torch.tensor(ANGIO),
                              'text': torch.randn(N, S_MAX, TXT_DIM), 'text_mask': torch.ones(N, S_MAX), 'text_sec': torch.zeros(N, S_MAX).long(), 'wear': torch.tensor(WEAR), 'y': torch.tensor(Y)}.items()}
_m = QM2H(DNUM, ECG_FEAT.shape[1], TXT_DIM)
_o = _m(_D0, torch.ones(8, 6))
print('parameters:', sum(p.numel() for p in _m.parameters()), '| logits', tuple(_o['logits'].shape), '| gate row sums', _o['gate'].sum(1)[:3].tolist())
del _m, _o, _D0

## 7. Federated toolkit (Section 3.4)
* **Toy lattice HE (Eq.23-24):** secret-key RLWE, `c = (-a*s + e + Delta*m, a)` over `Z_q[x]/(x^n+1)`; ciphertexts add component-wise, decryption of the sum equals the sum of plaintexts (additive homomorphism is all FedAvg needs once clients pre-multiply their update by the public weight `n_k/N`).
* **Robustness under encryption:** coordinate-wise trimmed mean needs comparisons, which additive HE cannot do. With `use_he=True` the server instead runs a *scalar-metadata* robust filter (drops clients whose update norm > 2x median) and then performs the encrypted weighted sum. With `use_he=False` the exact coordinate-wise trimmed mean (Eq.22) is used.
* **DP (Eq.25-26):** update clipping + Gaussian noise (client-level DP-FedAvg) with Renyi-DP accounting of the Gaussian mechanism.
* **Communication (Eq.27-28):** top-k sparsification with error feedback (residual memory).

In [ ]:
class ToyRLWE:
    SCALE = 2 ** 14
    def __init__(s, n=1024, q=2 ** 31, t=2 ** 17, sigma=3.2, seed=0):
        s.n, s.q, s.t, s.sigma = n, q, t, sigma; s.delta = q // t
        s.rng = np.random.default_rng(seed); s.sk = s.rng.integers(-1, 2, n).astype(np.int64)
    def _mul(s, a):                                          # negacyclic product a * sk mod (x^n + 1, q), exact in int64
        full = np.convolve(a, s.sk); out = full[:s.n].copy(); out[:s.n - 1] -= full[s.n:]
        return out % s.q
    def enc_vec(s, v):
        P = int(np.ceil(len(v) / s.n)); x = np.zeros(P * s.n); x[:len(v)] = v
        m = (np.rint(x * s.SCALE).astype(np.int64) % s.t).reshape(P, s.n)
        c0 = np.empty((P, s.n), np.int64); c1 = np.empty((P, s.n), np.int64)
        for j in range(P):
            a = s.rng.integers(0, s.q, s.n, dtype=np.int64); e = np.rint(s.rng.normal(0, s.sigma, s.n)).astype(np.int64)
            c0[j] = (-s._mul(a) + e + s.delta * m[j]) % s.q; c1[j] = a
        return c0, c1
    def dec_vec(s, c0, c1, length):
        out = np.empty(c0.shape, np.int64)
        for j in range(c0.shape[0]):
            v = (c0[j] + s._mul(c1[j])) % s.q; v = np.where(v > s.q // 2, v - s.q, v)
            mj = np.rint(v / s.delta).astype(np.int64) % s.t; out[j] = np.where(mj >= s.t // 2, mj - s.t, mj)
        return out.reshape(-1)[:length] / s.SCALE
    def ct_bytes(s, length):
        return 2 * int(np.ceil(length / s.n)) * s.n * 4          # two 31-bit words per coefficient

# homomorphic-addition self-test (Eq.24): Dec(Enc(a)+Enc(b)+Enc(c)) == a+b+c
_he = ToyRLWE(); _r = np.random.default_rng(1); _v = [_r.normal(0, 0.05, 3000) for _ in range(3)]
_cts = [_he.enc_vec(x) for x in _v]
_dec = _he.dec_vec(sum(c[0] for c in _cts) % _he.q, sum(c[1] for c in _cts) % _he.q, 3000)
print('HE additive-homomorphism max abs error: %.2e  (fixed-point resolution %.1e)' % (np.abs(_dec - sum(_v)).max(), 1 / ToyRLWE.SCALE))

# ---- Renyi DP accounting for the Gaussian mechanism (Eq.26): D_alpha(N(0,s^2)||N(1,s^2)) = alpha/(2 s^2), composed over T rounds
def dp_epsilon(sigma, T, delta, alphas=np.r_[np.arange(1.25, 10, 0.25), np.arange(10, 257, 4)]):
    if sigma <= 0:
        return float('inf')
    return float(np.min([T * a / (2 * sigma ** 2) + np.log(1 / delta) / (a - 1) for a in alphas]))

# ---- Aggregators (Eq.19-22, 35)
def aggregate(updates, weights, he):
    norms = np.array([np.linalg.norm(u) for u in updates]); info = {}
    if CFG.use_he:
        keep = norms <= 2.0 * np.median(norms) + 1e-12
        if keep.sum() == 0:
            keep[:] = True
        w = np.asarray(weights) * keep; w = w / w.sum()
        cts = [he.enc_vec(w[k] * updates[k]) for k in range(len(updates)) if keep[k]]
        avg = he.dec_vec(sum(c[0] for c in cts) % he.q, sum(c[1] for c in cts) % he.q, len(updates[0]))
        info = dict(kept=int(keep.sum()), ct_bytes=he.ct_bytes(len(updates[0])) * int(keep.sum()))
    else:
        U = np.stack(updates)
        if CFG.trim > 0 and len(U) > 2 * CFG.trim:
            avg = np.sort(U, 0)[CFG.trim:len(U) - CFG.trim].mean(0)                    # trimmed mean (Eq.22)
        else:
            avg = (np.asarray(weights)[:, None] * U).sum(0)
        info = dict(kept=len(U) - 2 * CFG.trim)
    return avg, info

class FedYogi:                                    # Eq.21 written for the pseudo-gradient Delta = mean(local - global)
    def __init__(s, n, dev, b1=0.9, b2=0.99):
        s.b1, s.b2, s.lr, s.tau = b1, b2, CFG.server_lr, CFG.yogi_tau
        s.m = torch.zeros(n, device=dev); s.v = torch.full((n,), s.tau ** 2, device=dev)
    def step(s, theta, delta):
        s.m = s.b1 * s.m + (1 - s.b1) * delta; d2 = delta * delta
        s.v = s.v - (1 - s.b2) * d2 * torch.sign(s.v - d2)
        return theta + s.lr * s.m / (s.v.sqrt() + s.tau)

# ---- Local optimisation: adaptive gradient clipping + Lookahead (Eq.36)
def agc_clip_(params, clip=0.1, eps=1e-3):
    for p in params:
        if p.grad is None:
            continue
        dims = tuple(range(1, p.ndim)) if p.ndim > 1 else None
        pn = (p.detach().norm(dim=dims, keepdim=True) if dims else p.detach().norm()).clamp_min(eps)
        gn = (p.grad.norm(dim=dims, keepdim=True) if dims else p.grad.norm()).clamp_min(1e-6)
        p.grad.mul_((pn * clip / gn).clamp(max=1.0))

class Lookahead:
    def __init__(s, opt, k=5, alpha=0.5):
        s.opt, s.k, s.alpha, s.n = opt, k, alpha, 0
        s.params = [p for g in opt.param_groups for p in g['params']]; s.slow = [p.detach().clone() for p in s.params]
    def zero_grad(s):
        s.opt.zero_grad(set_to_none=True)
    def step(s):
        s.opt.step(); s.n += 1
        if s.n % s.k == 0:
            for sp, p in zip(s.slow, s.params):
                sp.add_(p.detach() - sp, alpha=s.alpha); p.data.copy_(sp)

get_vec = lambda m: torch.nn.utils.parameters_to_vector(m.parameters()).detach().clone()
set_vec = lambda m, v: torch.nn.utils.vector_to_parameters(v, m.parameters())

## 8. Training & evaluation routines

In [ ]:
def cls_metrics(y, proba):
    Cc = proba.shape[1]; pred = proba.argmax(1)
    cm = confusion_matrix(y, pred, labels=list(range(Cc))); n_ = cm.sum()
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = n_ - tp - fp - fn
    dv = lambda a, b: np.where(b > 0, a / np.maximum(b, 1), 0.0)
    prec, rec, spec, npv = dv(tp, tp + fp), dv(tp, tp + fn), dv(tn, tn + fp), dv(tn, tn + fn)
    f1 = dv(2 * prec * rec, prec + rec); oh = np.eye(Cc)[y]
    return {'Accuracy': 100 * tp.sum() / n_, 'Precision': 100 * prec.mean(), 'Recall': 100 * rec.mean(), 'F1': 100 * f1.mean(),
            'Specificity': 100 * spec.mean(), 'MCC': 100 * matthews_corrcoef(y, pred), 'NPV': 100 * npv.mean(),
            'FPR': 100 * (1 - spec).mean(), 'FNR': 100 * (1 - rec).mean(),
            'RMSE': float(np.sqrt(((proba - oh) ** 2).mean())), 'MAE': float(np.abs(proba - oh).mean()),
            'R2': float(1 - ((proba - oh) ** 2).sum() / ((oh - oh.mean(0)) ** 2).sum())}

def ece_score(proba, y, bins=10):
    conf = proba.max(1); pred = proba.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def take(D, bi):
    return {k: v[bi] for k, v in D.items() if k != 'y'}

def modality_dropout(mask, p):
    keep = (torch.rand_like(mask) > p).float() * mask
    none = keep.sum(1) == 0
    keep[none] = mask[none]
    return keep

@torch.no_grad()
def predict(model, D, mask, idx, mc=0, bs=128, want_gate=False):
    dev = next(model.parameters()).device; idx = torch.as_tensor(np.asarray(idx), device=dev)
    model.eval()
    if mc > 0:                                              # Monte-Carlo dropout = approximate posterior over parameters (Eq.38)
        for m in model.modules():
            if isinstance(m, nn.Dropout):
                m.train()
    P, G = [], []
    for i in range(0, len(idx), bs):
        bi = idx[i:i + bs]; b = take(D, bi); mk = mask[bi]; ps = 0
        for _ in range(max(mc, 1)):
            o = model(b, mk); ps = ps + o['logits'].softmax(-1)
        P.append((ps / max(mc, 1)).cpu()); G.append(o['gate'].cpu())
    model.eval()
    P = torch.cat(P).numpy()
    return (P, torch.cat(G).numpy()) if want_gate else P

def local_train(model, D, mask, idx, epochs, cw):
    model.train()
    opt = Lookahead(torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-3), k=4, alpha=0.5)
    y = D['y']; tot, nb = 0.0, 0
    for _ in range(epochs):
        perm = idx[torch.randperm(len(idx), device=idx.device)]
        for i in range(0, len(perm), CFG.batch):
            bi = perm[i:i + CFG.batch]
            if len(bi) < 4:
                continue
            b = take(D, bi); mk = modality_dropout(mask[bi], CFG.mod_drop); yb = y[bi]
            o = model(b, mk)
            loss = F.cross_entropy(o['logits'], yb, weight=cw, label_smoothing=0.05)
            ce_m = torch.stack([F.cross_entropy(o['el'][:, m], yb, reduction='none') for m in range(6)], 1)
            loss = loss + 0.3 * (ce_m * mk).sum() / mk.sum().clamp_min(1)                  # per-expert auxiliary loss
            gm = (o['gate'] * mk).mean(0); gm = gm / gm.sum().clamp_min(1e-9)
            loss = loss + 0.01 * (gm * torch.log(gm + 1e-9)).sum()                            # expert-usage balance
            opt.zero_grad(); loss.backward(); agc_clip_(model.parameters(), CFG.agc); opt.step()
            tot += loss.item(); nb += 1
    return tot / max(nb, 1)

def run_federated(D, mask, tr_idx, va_idx, te_idx, client_idx, rounds, verbose=True, **model_kw):
    dev = torch.device(CFG.device)
    Dd = {k: v.to(dev) for k, v in D.items()}; mk = mask.to(dev)
    gm, lm = build_model(D, **model_kw).to(dev), build_model(D, **model_kw).to(dev)
    n = sum(p.numel() for p in gm.parameters())
    resid = [torch.zeros(n, device=dev) for _ in client_idx]; yogi = FedYogi(n, dev)
    he = ToyRLWE(seed=CFG.seed) if CFG.use_he else None
    cnt = np.bincount(Y[tr_idx], minlength=C); cw = torch.tensor(len(tr_idx) / (C * np.maximum(cnt, 1)), dtype=torch.float32, device=dev)
    weights = np.array([len(c) for c in client_idx], float); weights /= weights.sum()
    hist = dict(round=[], loss=[], val=[], test=[], agg_time=[]); best = (-1, None, 0)
    kk = max(1, int(CFG.topk_frac * n))
    for r in range(rounds):
        g = get_vec(gm); ups, losses = [], []
        for k, ci in enumerate(client_idx):
            set_vec(lm, g.clone())
            losses.append(local_train(lm, Dd, mk, torch.as_tensor(ci, device=dev), CFG.local_epochs, cw))
            delta = get_vec(lm) - g
            if CFG.byzantine == k:
                delta = -3 * delta                                                          # simulated malicious client
            msg = delta + resid[k]                                                          # error feedback (Eq.28)
            sp = torch.zeros_like(msg); ti = msg.abs().topk(kk).indices; sp[ti] = msg[ti]   # top-k sparsification (Eq.27)
            sp = sp * min(1.0, CFG.dp_clip / (sp.norm().item() + 1e-12))                     # clip
            resid[k] = msg - sp
            ups.append((sp + torch.randn_like(sp) * (CFG.dp_noise * CFG.dp_clip)).double().cpu().numpy())   # Gaussian mechanism (Eq.25)
        t0 = time.time(); avg, info = aggregate(ups, weights, he); hist['agg_time'].append(time.time() - t0)
        set_vec(gm, yogi.step(g, torch.tensor(avg, dtype=torch.float32, device=dev)).clone())
        pv = predict(gm, Dd, mk, va_idx); pt = predict(gm, Dd, mk, te_idx)
        av = float((pv.argmax(1) == Y[va_idx]).mean()); at = float((pt.argmax(1) == Y[te_idx]).mean())
        hist['round'].append(r + 1); hist['loss'].append(float(np.mean(losses))); hist['val'].append(av); hist['test'].append(at)
        if av >= best[0]:
            best = (av, copy.deepcopy(gm.state_dict()), r + 1)
        if verbose and (r % 2 == 1 or r == rounds - 1):
            print(f'round {r + 1:2d}/{rounds} loss {np.mean(losses):.3f} | val acc {av:.3f} | test acc {at:.3f} | agg {hist["agg_time"][-1]:.1f}s | {info}')
    gm.load_state_dict(best[1]); gm.eval()
    plain = 4 * n; sparse = 8 * kk
    return dict(model=gm, hist=hist, best_round=best[2], n_params=n, eps=dp_epsilon(CFG.dp_noise, rounds, CFG.dp_delta),
                comm=dict(dense_float32=plain, sparse_plain=sparse, he_ciphertext=(he.ct_bytes(n) if he else None)), Dd=Dd, mk=mk)

## 9. Main experiment: non-IID federated training with missing modalities

In [ ]:
tr_all, te_idx = train_test_split(np.arange(N), test_size=0.2, stratify=Y, random_state=CFG.seed)
tr_idx, va_idx = train_test_split(tr_all, test_size=0.12, stratify=Y[tr_all], random_state=CFG.seed)
clients = dirichlet_split(tr_idx, Y, CFG.n_clients, alpha=1.0, seed=CFG.seed)
client_of = -np.ones(N, int)
for k, c in enumerate(clients):
    client_of[c] = k
MASK = torch.tensor(build_masks(N, client_of, CFG.n_clients, CFG.seed))
D, ART = make_features(tr_idx)
print('train/val/test:', len(tr_idx), len(va_idx), len(te_idx))
print(pd.DataFrame({f'client{k}': [len(c)] + np.bincount(Y[c], minlength=3).tolist() +
                    (1 - MASK[c].mean(0)).numpy().round(2).tolist() for k, c in enumerate(clients)},
                   index=['n'] + [f'class_{r}' for r in RISK_NAMES] + [f'missing_{m}' for m in CFG.MODS]).T)

t0 = time.time()
RES = run_federated(D, MASK, tr_idx, va_idx, te_idx, clients, CFG.rounds)
print(f'\nbest round {RES["best_round"]} | params {RES["n_params"]:,} | wall time {time.time() - t0:.0f}s')
print('communication / client / round (bytes):', RES['comm'])
print('(epsilon, delta) = (%.3g, %g) at noise multiplier %.4g -> see note 4: tiny noise => no meaningful formal privacy' % (RES['eps'], CFG.dp_delta, CFG.dp_noise))
MODEL = RES['model']; Dd, MKd = RES['Dd'], RES['mk']

## 10. Results

In [ ]:
P_te, G_te = predict(MODEL, Dd, MKd, te_idx, want_gate=True); y_te = Y[te_idx]
full_m = cls_metrics(y_te, P_te)
bin_m = cls_metrics((y_te > 0).astype(int), np.stack([P_te[:, 0], P_te[:, 1:].sum(1)], 1))
print('Q-M2H, natural missing-modality pattern, 3-class risk:'); print(pd.Series(full_m).round(2).to_string())
print('\nDerived binary task (low vs medium/high):'); print(pd.Series(bin_m).round(2)[['Accuracy', 'Precision', 'Recall', 'F1', 'Specificity', 'MCC', 'FPR', 'FNR']].to_string())

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(RES['hist']['round'], RES['hist']['val'], label='val acc'); ax[0].plot(RES['hist']['round'], RES['hist']['test'], label='test acc')
ax[0].set_xlabel('federated round'); ax[0].set_ylabel('accuracy'); ax[0].legend(); ax[0].set_title('Federated training')
ax[1].plot(RES['hist']['round'], RES['hist']['loss'], 'r'); ax[1].set_title('mean local loss'); ax[1].set_xlabel('federated round')
cm = confusion_matrix(y_te, P_te.argmax(1), labels=[0, 1, 2]); ax[2].imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax[2].text(j, i, cm[i, j], ha='center', va='center')
ax[2].set_xticks(range(3)); ax[2].set_xticklabels(RISK_NAMES); ax[2].set_yticks(range(3)); ax[2].set_yticklabels(RISK_NAMES)
ax[2].set_title('Confusion matrix (test)'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('true')
plt.tight_layout(); plt.savefig(f'{CFG.out_dir}/fig_training.png', dpi=130); plt.show()

# per-modality confusion matrices (analogue of Fig. 9): only ONE modality visible at inference
def mask_only(m):
    mk = torch.zeros(N, 6, device=MKd.device); mk[:, m] = 1; return mk
fig, ax = plt.subplots(1, 6, figsize=(22, 3.6)); single = {}
for m, nm in enumerate(CFG.MODS):
    Pm = predict(MODEL, Dd, mask_only(m), te_idx); single[nm] = Pm
    cmx = confusion_matrix(y_te, Pm.argmax(1), labels=[0, 1, 2]); ax[m].imshow(cmx, cmap='Blues')
    for i in range(3):
        for j in range(3):
            ax[m].text(j, i, cmx[i, j], ha='center', va='center', fontsize=8)
    ax[m].set_title(f'{nm}-only\nacc {100 * (Pm.argmax(1) == y_te).mean():.1f}%', fontsize=9); ax[m].set_xticks(range(3)); ax[m].set_yticks(range(3))
plt.tight_layout(); plt.savefig(f'{CFG.out_dir}/fig_confusion_per_modality.png', dpi=130); plt.show()

# which experts does the entropy-aware router use for each risk class?
gate_df = pd.DataFrame(G_te, columns=CFG.MODS); gate_df['class'] = [RISK_NAMES[c] for c in y_te]
gm_ = gate_df.groupby('class')[CFG.MODS].mean().loc[RISK_NAMES]
plt.figure(figsize=(7, 2.8)); plt.imshow(gm_.values, cmap='viridis', aspect='auto'); plt.colorbar(label='mean gate weight')
plt.xticks(range(6), CFG.MODS, rotation=30); plt.yticks(range(3), RISK_NAMES); plt.title('Dynamic expert routing (Eq.34)')
plt.tight_layout(); plt.savefig(f'{CFG.out_dir}/fig_gating.png', dpi=130); plt.show()

## 11. Bayesian-calibrated risk probabilities (Section 3.7.1, Eq.38)
The posterior predictive `p(y|x) = E_{theta~p(theta|D)} p(y|x,theta)` is approximated with **Monte-Carlo dropout** (T samples); a temperature fitted on the validation split then removes residual over/under-confidence.

In [ ]:
Pv_mc = predict(MODEL, Dd, MKd, va_idx, mc=CFG.mc_samples); Pt_mc = predict(MODEL, Dd, MKd, te_idx, mc=CFG.mc_samples)
y_va = Y[va_idx]
def _nll(T, P, y):
    L_ = np.log(P + 1e-9) / T; L_ = L_ - L_.max(1, keepdims=True); Q = np.exp(L_); Q /= Q.sum(1, keepdims=True)
    return -np.log(Q[np.arange(len(y)), y] + 1e-9).mean()
T_opt = minimize_scalar(lambda T: _nll(T, Pv_mc, y_va), bounds=(0.3, 5), method='bounded').x
def _scale(P, T):
    L_ = np.log(P + 1e-9) / T; L_ = L_ - L_.max(1, keepdims=True); Q = np.exp(L_); return Q / Q.sum(1, keepdims=True)
Pt_cal = _scale(Pt_mc, T_opt)
print(f'temperature = {T_opt:.3f}')
print('test ECE  single pass: %.4f | MC-dropout: %.4f | MC-dropout + temperature: %.4f' % (ece_score(P_te, y_te), ece_score(Pt_mc, y_te), ece_score(Pt_cal, y_te)))
print('test NLL  single pass: %.4f | calibrated: %.4f' % (-np.log(P_te[np.arange(len(y_te)), y_te] + 1e-9).mean(), -np.log(Pt_cal[np.arange(len(y_te)), y_te] + 1e-9).mean()))
unc = -(Pt_cal * np.log(Pt_cal + 1e-9)).sum(1)
print('predictive entropy: correct = %.3f, wrong = %.3f' % (unc[Pt_cal.argmax(1) == y_te].mean(), unc[Pt_cal.argmax(1) != y_te].mean() if (Pt_cal.argmax(1) != y_te).any() else float('nan')))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a_, (P_, nm) in zip(ax, [(P_te, 'single pass'), (Pt_cal, 'MC-dropout + temperature')]):
    conf = P_.max(1); cor = (P_.argmax(1) == y_te); bins = np.linspace(0, 1, 6)
    xs = [(conf[(conf > lo) & (conf <= hi)].mean(), cor[(conf > lo) & (conf <= hi)].mean()) for lo, hi in zip(bins[:-1], bins[1:]) if ((conf > lo) & (conf <= hi)).sum() > 0]
    a_.plot([0, 1], [0, 1], 'k--'); a_.plot(*zip(*xs), 'o-'); a_.set_title(f'Reliability: {nm}'); a_.set_xlabel('confidence'); a_.set_ylabel('accuracy')
plt.tight_layout(); plt.savefig(f'{CFG.out_dir}/fig_calibration.png', dpi=130); plt.show()

## 12. Explainability — SHAP-QM (Section 3.7.2, Eq.39)
* **Modality level:** *exact* Shapley values over the patient's available modalities (<=64 coalitions). A coalition's value is the model's probability of the predicted class when the other modalities are **masked** (the MMA module makes this a legitimate model input). Each Shapley value is then re-weighted by a *quantum influence factor* `kappa_m` = share of QCCA attention that modality's quantum token receives (normalised to mean 1). Note: the re-weighting deliberately trades Shapley's efficiency axiom for quantum-awareness; raw values are also returned.
* **Feature level:** Monte-Carlo permutation Shapley on the clinical features (grouped, e.g. all `cp_*` dummies = one feature) with other modalities fixed.

In [ ]:
def modality_shapley(model, Dd, i, mask_row):
    dev = mask_row.device; M = 6; avail = [m for m in range(M) if mask_row[m] > 0]; n = len(avail)
    subs = list(itertools.product([0, 1], repeat=n)); S = len(subs); mk = torch.zeros(S, M, device=dev)
    for si, sub in enumerate(subs):
        for a, m in zip(sub, avail):
            mk[si, m] = float(a)
    b = {k: v[i:i + 1].expand(S, *v.shape[1:]).contiguous() for k, v in Dd.items() if k != 'y'}
    with torch.no_grad():
        pr = model(b, mk)['logits'].softmax(-1)
        tgt = int(pr[-1].argmax()); val = {sub: pr[si, tgt].item() for si, sub in enumerate(subs)}
        model({k: v[i:i + 1] for k, v in Dd.items() if k != 'y'}, mask_row.unsqueeze(0))
        att = model.qcca.last_att[0].mean(0)[avail]; kappa = (att / att.mean()).cpu().numpy()
    raw, qw = {}, {}
    for j, m in enumerate(avail):
        tot = 0.0
        for sub in subs:
            if sub[j] == 1:
                continue
            s_ = sum(sub); w = math.factorial(s_) * math.factorial(n - s_ - 1) / math.factorial(n)
            s1 = list(sub); s1[j] = 1; tot += w * (val[tuple(s1)] - val[sub])
        raw[CFG.MODS[m]] = tot; qw[CFG.MODS[m]] = tot * float(kappa[j])
    return tgt, qw, raw

def numeric_shapley(model, Dd, i, mask_row, tgt, bg, n_perm=30, seed=0):
    rng = np.random.default_rng(seed); names = list(NUM_GROUPS); G = len(names)
    x = Dd['num'][i]; phi = np.zeros(G)
    b0 = {k: v[i:i + 1].expand(G + 1, *v.shape[1:]).contiguous() for k, v in Dd.items() if k not in ('y', 'num')}
    mk = mask_row.unsqueeze(0).expand(G + 1, -1)
    for _ in range(n_perm):
        order = rng.permutation(G); cur = bg.clone(); rows = [cur.clone()]
        for g in order:
            cur[NUM_GROUPS[names[g]]] = x[NUM_GROUPS[names[g]]]; rows.append(cur.clone())
        b = dict(b0); b['num'] = torch.stack(rows)
        with torch.no_grad():
            pr = model(b, mk)['logits'].softmax(-1)[:, tgt].cpu().numpy()
        for pos, g in enumerate(order):
            phi[g] += pr[pos + 1] - pr[pos]
    return dict(zip(names, phi / n_perm))

bg_num = D['num'][tr_idx].mean(0).to(Dd['num'].device)
cands = [i for i in te_idx if Y[i] == 2 and MKd[i].sum() >= 4 and P_te[list(te_idx).index(i)].argmax() == 2]
PAT = int(cands[0] if cands else te_idx[0])
tgt, qw, rawphi = modality_shapley(MODEL, Dd, PAT, MKd[PAT])
nphi = numeric_shapley(MODEL, Dd, PAT, MKd[PAT], tgt, bg_num)
print(f'patient #{PAT}: true class {RISK_NAMES[Y[PAT]]}, predicted {RISK_NAMES[tgt]}, available modalities: {[CFG.MODS[m] for m in range(6) if MKd[PAT, m] > 0]}')
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].barh(list(qw), list(qw.values()), color=['tab:red' if v < 0 else 'tab:blue' for v in qw.values()]); ax[0].set_title('SHAP-QM per modality (effect on P[%s])' % RISK_NAMES[tgt])
srt = sorted(nphi.items(), key=lambda kv: abs(kv[1]))[-9:]
ax[1].barh([k for k, _ in srt], [v for _, v in srt], color=['tab:red' if v < 0 else 'tab:blue' for _, v in srt]); ax[1].set_title('Clinical-feature Shapley values')
plt.tight_layout(); plt.savefig(f'{CFG.out_dir}/fig_shap_qm.png', dpi=130); plt.show()

## 13. Counterfactual interventions (Section 3.7.3, Eq.40)
Gradient-guided minimal change of the **actionable** clinical features only (systolic BP and cholesterol may only decrease, peak heart rate / fitness may only increase), within physiological bounds, other modalities fixed: `min_delta CE(f(x+delta), low-risk) + lambda*||delta||_1`.

In [ ]:
ACT = {'trestbps': 1, 'chol': 2, 'thalach': 3}       # column index in the RRG-normalised numeric vector
ACT_DIR = {'trestbps': -1, 'chol': -1, 'thalach': +1}
ACT_BOUND = {'trestbps': (100, None), 'chol': (150, None), 'thalach': (None, float(np.percentile(RAWC[:, 3], 95)))}

def counterfactual(model, Dd, i, mask_row, target=0, steps=300, lam=0.03, lr=0.05):
    model.eval(); dev = Dd['num'].device
    x0 = Dd['num'][i:i + 1].clone(); cols = [ACT[k] for k in ACT]
    lo = torch.zeros(3, device=dev); hi = torch.zeros(3, device=dev)
    for j, k in enumerate(ACT):
        z0 = x0[0, ACT[k]].item(); r_ = ART['rrgs'][ACT[k]]; bl, bh = ACT_BOUND[k]
        if ACT_DIR[k] < 0:
            lo[j] = (float(r_.transform(np.array([bl]))[0]) - z0) if bl is not None else -3 - z0; lo[j] = min(lo[j].item(), 0.0)
        else:
            hi[j] = (float(r_.transform(np.array([bh]))[0]) - z0) if bh is not None else 3 - z0; hi[j] = max(hi[j].item(), 0.0)
    delta = torch.zeros(3, device=dev, requires_grad=True); opt = torch.optim.Adam([delta], lr=lr)
    b0 = {k: v[i:i + 1] for k, v in Dd.items() if k not in ('y', 'num')}; mk = mask_row.unsqueeze(0); tt = torch.tensor([target], device=dev)
    for _ in range(steps):
        x = x0.clone(); x[0, cols] = x[0, cols] + delta; b = dict(b0); b['num'] = x
        loss = F.cross_entropy(model(b, mk)['logits'], tt) + lam * delta.abs().sum()
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            delta.copy_(torch.max(torch.min(delta, hi), lo))
    x = x0.clone(); x[0, cols] = x[0, cols] + delta.detach(); b = dict(b0); b['num'] = x
    with torch.no_grad():
        p_new = model(b, mk)['logits'].softmax(-1)[0].cpu().numpy(); p_old = model({**b0, 'num': x0}, mk)['logits'].softmax(-1)[0].cpu().numpy()
    rows = []
    for j, k in enumerate(ACT):
        z_old = x0[0, ACT[k]].item(); z_new = x[0, ACT[k]].item(); r_ = ART['rrgs'][ACT[k]]
        rows.append((k, float(r_.inverse(z_old)), float(r_.inverse(z_new))))
    return p_old, p_new, pd.DataFrame(rows, columns=['feature', 'current', 'counterfactual'])

shown = 0
for i in [j for j in te_idx if Y[j] >= 1][:12]:
    p_old, p_new, tab = counterfactual(MODEL, Dd, int(i), MKd[int(i)])
    if p_new.argmax() == 0 or shown == 0:
        print(f'patient #{i}: P(low,med,high) {np.round(p_old, 2)} -> {np.round(p_new, 2)}'); print(tab.round(1).to_string(index=False)); print(); shown += 1
    if shown >= 3:
        break

## 14. Safe meta-RL recommender (Section 3.7.4, Eq.41)
Environment = a *simulated* 6-month intervention programme on the three actionable features (diet, statin, antihypertensive, supervised exercise, intensive combined plan). The reward is the drop of a **risk surrogate distilled from Q-M2H** (logistic regression on Q-M2H's own predictions) minus action cost; the safety cost counts hypotension (BP < 100) and unsupervised exertion in very-high-risk states. Training = **Reptile meta-learning** across patients with different (unobserved) adherence, with a **Lagrangian** that enforces `E[cost] <= d`. The treatment-effect table below is an *illustrative assumption*, not clinical evidence — replace it with effect sizes from trials / your cohort before any real use.

In [ ]:
ACTIONS = ['monitor only', 'diet + sodium reduction', 'statin therapy', 'antihypertensive titration', 'supervised exercise', 'intensive combined plan']
EFFECT = np.array([[0, 0, 0], [-3, -10, 0], [0, -35, 0], [-10, 0, 0], [-2, 0, 6], [-12, -40, 4]], float)   # per month: d(trestbps), d(chol), d(thalach)
COST = np.array([0, .02, .05, .05, .04, .12]); HZ = 6; SAFE_D = 0.10
LO3 = np.array([85., 120., 70.]); HI3 = np.array([200., 400., float(RAWC[:, 3].max())])
MU3, SD3 = RAWC[:, 1:4].mean(0), RAWC[:, 1:4].std(0)

# risk surrogate distilled from the trained Q-M2H (soft labels via duplicated weighted samples)
P_all = predict(MODEL, Dd, MKd, np.arange(N)); Xn = D['num'].numpy(); p_nl = 1 - P_all[:, 0]
sur = LogisticRegression(C=1.0, max_iter=1000).fit(np.vstack([Xn, Xn]), np.r_[np.zeros(N), np.ones(N)], sample_weight=np.r_[1 - p_nl, p_nl])
print('surrogate vs Q-M2H P(at-risk) correlation: %.3f' % np.corrcoef(sur.predict_proba(Xn)[:, 1], p_nl)[0, 1])

class Env:
    def __init__(s, i, adherence, seed=0):
        s.x = Xn[i].copy(); s.raw0 = RAWC[i, 1:4].copy(); s.adh = adherence; s.rng = np.random.default_rng(seed)
    def risk(s, raw):
        X = np.tile(s.x, (len(raw), 1))
        for j, col in enumerate([1, 2, 3]):
            X[:, col] = ART['rrgs'][col].transform(raw[:, j])
        return sur.predict_proba(X)[:, 1]
    def step(s, raw, risk, a):
        eff = EFFECT[a] * s.adh * (1 + s.rng.normal(0, 0.2, raw.shape)); new = np.clip(raw + eff, LO3, HI3); rn = s.risk(new)
        return new, rn, (risk - rn) * 10 - COST[a], (new[:, 0] < 100).astype(float) + ((a >= 4) & (risk > 0.85)).astype(float)

class Policy(nn.Module):
    def __init__(s):
        super().__init__(); s.net = nn.Sequential(nn.Linear(5, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, len(ACTIONS)))
    def forward(s, x):
        return s.net(x)

def rollout(policy, env, B=8, greedy=False):
    raw = np.tile(env.raw0, (B, 1)); risk = env.risk(raw); R = np.zeros(B); Cc = np.zeros(B); lps = []; acts = []
    for t in range(HZ):
        st = torch.tensor(np.concatenate([(raw - MU3) / SD3, risk[:, None], np.full((B, 1), t / HZ)], 1), dtype=torch.float32)
        dist = torch.distributions.Categorical(logits=policy(st)); a = dist.logits.argmax(-1) if greedy else dist.sample()
        lps.append(dist.log_prob(a)); acts.append(a.numpy())
        raw, risk, r, c = env.step(raw, risk, a.numpy()); R += r; Cc += c
    return torch.stack(lps).sum(0), R, Cc, raw, risk, np.stack(acts, 1)

def inner_adapt(policy, env, steps, lam, B=8, lr=0.05):
    p = copy.deepcopy(policy); opt = torch.optim.SGD(p.parameters(), lr=lr)
    for _ in range(steps):
        lp, R, Cc, _, _, _ = rollout(p, env, B); adv = torch.tensor(R - lam * Cc, dtype=torch.float32); adv = adv - adv.mean()
        loss = -(adv * lp).mean(); opt.zero_grad(); loss.backward(); opt.step()
    return p

at_risk = np.where(Y[tr_idx] >= 1)[0]; at_risk = tr_idx[at_risk]
rl_rng = np.random.default_rng(CFG.seed); policy = Policy(); lam = 0.5; log = []
t0 = time.time()
for it in range(150):
    env = Env(int(rl_rng.choice(at_risk)), rl_rng.uniform(0.3, 1.0), seed=it)
    p_task = inner_adapt(policy, env, 3, lam)
    with torch.no_grad():
        for a_, b_ in zip(policy.parameters(), p_task.parameters()):
            a_ += 0.3 * (b_ - a_)                                                 # Reptile outer step
    _, R, Cc, _, _, _ = rollout(p_task, env, 8); lam = max(0.0, lam + 0.1 * (Cc.mean() - SAFE_D)); log.append((R.mean(), Cc.mean(), lam))
print(f'meta-training {time.time() - t0:.0f}s | last-30 mean return {np.mean([l[0] for l in log[-30:]]):.2f}, mean cost {np.mean([l[1] for l in log[-30:]]):.3f}, lambda {lam:.2f}')

# evaluation on held-out patients: random policy vs meta-initialised + 5-step adaptation
ev_rng = np.random.default_rng(123); rows = []
for i in [int(j) for j in te_idx if Y[j] >= 1][:20]:
    env = Env(i, ev_rng.uniform(0.3, 1.0), seed=int(i)); rp = Policy()
    for name, pol in [('random init', rp), ('meta + adapt', inner_adapt(policy, env, 5, lam))]:
        _, R, Cc, _, _, _ = rollout(pol, env, 16); rows.append((name, R.mean(), Cc.mean()))
print(pd.DataFrame(rows, columns=['policy', 'return', 'safety cost']).groupby('policy').mean().round(3))

# personalised plans for three high-risk test patients, verified with the *real* Q-M2H
for i in [int(j) for j in te_idx if Y[j] == 2][:3]:
    env = Env(i, 0.7, seed=1); pol = inner_adapt(policy, env, 5, lam)
    _, R, Cc, raw_f, risk_f, acts = rollout(pol, env, 1, greedy=True)
    b = {k: v[i:i + 1] for k, v in Dd.items() if k != 'y'}; x_new = Dd['num'][i:i + 1].clone()
    for j, col in enumerate([1, 2, 3]):
        x_new[0, col] = float(ART['rrgs'][col].transform(raw_f[:, j])[0])
    with torch.no_grad():
        p0 = MODEL(b, MKd[i:i + 1])['logits'].softmax(-1)[0].cpu().numpy(); p1 = MODEL({**b, 'num': x_new}, MKd[i:i + 1])['logits'].softmax(-1)[0].cpu().numpy()
    print(f'\npatient #{i}: plan by month -> {[ACTIONS[a] for a in acts[0]]}')
    print(f'   BP/chol/HR {env.raw0.round(0)} -> {raw_f[0].round(0)} | Q-M2H P(low) {p0[0]:.2f} -> {p1[0]:.2f} | safety cost {Cc[0]:.0f}')

## 15. Baselines and per-modality comparison (Tables 6-11 analogue)
Baselines see the **complete** data of each modality (no missingness) — an advantage over Q-M2H, which is evaluated with only that modality visible after being trained on incomplete multimodal data. ResNet = compact residual network on the modality's feature vector (a full ResNet18 is not meaningful for 20-150-dim vectors); LSTM reads the feature vector as a short sequence; XGBoost falls back to sklearn HistGradientBoosting if xgboost is unavailable.

In [ ]:
class TabResNet(nn.Module):
    def __init__(s, din, Cc=3, w=64, nb=3):
        super().__init__(); s.inp = nn.Linear(din, w)
        s.blocks = nn.ModuleList([nn.Sequential(nn.LayerNorm(w), nn.Linear(w, w), nn.ReLU(), nn.Dropout(0.1), nn.Linear(w, w)) for _ in range(nb)])
        s.out = nn.Sequential(nn.LayerNorm(w), nn.ReLU(), nn.Linear(w, Cc))
    def forward(s, x):
        h = s.inp(x)
        for b in s.blocks:
            h = h + b(h)
        return s.out(h)

class SeqLSTM(nn.Module):
    def __init__(s, din, Cc=3, hid=32, steps=16):
        super().__init__(); s.steps = steps; s.chunk = math.ceil(din / steps); s.pad = s.steps * s.chunk - din
        s.lstm = nn.LSTM(s.chunk, hid, batch_first=True); s.out = nn.Linear(hid, Cc)
    def forward(s, x):
        x = F.pad(x, (0, s.pad)).view(x.size(0), s.steps, s.chunk); _, (h, _) = s.lstm(x); return s.out(h[-1])

def train_torch_baseline(net, Xtr, ytr, Xte, epochs, lr=2e-3, bs=32):
    dev = torch.device(CFG.device); mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xt = torch.tensor((Xtr - mu) / sd, dtype=torch.float32, device=dev); yt = torch.tensor(ytr, device=dev); net = net.to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
    for _ in range(epochs):
        net.train(); perm = torch.randperm(len(Xt), device=dev)
        for i in range(0, len(perm), bs):
            bi = perm[i:i + bs]
            if len(bi) < 2:
                continue
            loss = F.cross_entropy(net(Xt[bi]), yt[bi]); opt.zero_grad(); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        return net(torch.tensor((Xte - mu) / sd, dtype=torch.float32, device=dev)).softmax(-1).cpu().numpy()

def make_xgb():
    return XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, eval_metric='mlogloss', verbosity=0) if HAS_XGB \
        else HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1, max_iter=150)

def fit_baselines(Xtr, ytr, Xte):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6; out = {}
    out['ResNet'] = train_torch_baseline(TabResNet(Xtr.shape[1]), Xtr, ytr, Xte, CFG.baseline_epochs)
    out['LSTM'] = train_torch_baseline(SeqLSTM(Xtr.shape[1]), Xtr, ytr, Xte, CFG.baseline_epochs)
    out['XGBoost'] = make_xgb().fit(Xtr, ytr).predict_proba(Xte)
    out['SVM'] = SVC(C=3, gamma='scale', probability=True, random_state=0).fit((Xtr - mu) / sd, ytr).predict_proba((Xte - mu) / sd)
    return out

def modality_features(D_):
    n_ = len(D_['y']); ds = lambda a, k: a[:, :(a.shape[1] // k) * k].reshape(a.shape[0], -1, k).mean(2)
    w = D_['wear'].numpy(); txt = ((D_['text'] * D_['text_mask'].unsqueeze(-1)).sum(1) / (D_['text_mask'].sum(1, keepdim=True) + 1e-6)).numpy()
    return {'numerical': D_['num'].numpy(),
            'ecg': np.concatenate([D_['ecg_feat'].numpy(), ds(D_['ecg_sig'].numpy(), 12)], 1),
            'echo': F.avg_pool2d(D_['echo'], 2).flatten(1).numpy(), 'angio': F.avg_pool2d(D_['angio'], 2).flatten(1).numpy(),
            'text': txt, 'wearable': np.concatenate([w.reshape(n_, 12, 4, 3).mean(2).reshape(n_, -1), w.mean(1), w.std(1)], 1)}

def early_fusion_X(feats, mask):
    mk = mask.cpu().numpy()
    return np.concatenate([feats[nm] * mk[:, m:m + 1] for m, nm in enumerate(CFG.MODS)] + [mk], 1)

def evaluate_all(D_, mask_cpu, tr, te, model, Dd_, mk_dev):
    feats = modality_features(D_); out = {}
    for m, nm in enumerate(CFG.MODS):
        mm = torch.zeros(N, 6, device=mk_dev.device); mm[:, m] = 1
        o = {'Proposed': predict(model, Dd_, mm, te)}; o.update(fit_baselines(feats[nm][tr], Y[tr], feats[nm][te])); out[nm] = o
    Xef = early_fusion_X(feats, mask_cpu)
    out['multimodal'] = {'Proposed': predict(model, Dd_, mk_dev, te), 'XGB early-fusion': make_xgb().fit(Xef[tr], Y[tr]).predict_proba(Xef[te])}
    return out

t0 = time.time(); EV = evaluate_all(D, MASK, tr_idx, te_idx, MODEL, Dd, MKd); print(f'baselines done in {time.time() - t0:.0f}s')
TABLES = {}
for nm, o in EV.items():
    TABLES[nm] = pd.DataFrame({meth: cls_metrics(y_te, P) for meth, P in o.items()}).T.round(2)
    print(f'\n=== {nm} ==='); print(TABLES[nm][['Accuracy', 'Precision', 'Recall', 'F1', 'Specificity', 'MCC', 'NPV', 'FPR', 'FNR', 'RMSE', 'MAE', 'R2']].to_string())
    TABLES[nm].to_csv(f'{CFG.out_dir}/table_{nm}.csv')

## 16. Five-fold cross-validation and paired t-tests (Figures 3-8, Tables 12-14 analogue)
Each fold re-splits patients, re-builds the non-IID clients and modality masks, and re-fits every fitted preprocessing step on that fold's training data. With only 5 folds the t-tests have 4 degrees of freedom — treat p-values as indicative. Set `CFG.run_cv=False` to skip (takes the longest).

In [ ]:
CV = {nm: {} for nm in CFG.MODS + ['multimodal']}
if CFG.run_cv:
    skf = StratifiedKFold(CFG.n_folds, shuffle=True, random_state=CFG.seed)
    for f, (tr, te) in enumerate(skf.split(np.zeros(N), Y)):
        t0 = time.time(); trf, vaf = train_test_split(tr, test_size=0.12, stratify=Y[tr], random_state=f)
        cl = dirichlet_split(trf, Y, CFG.n_clients, seed=CFG.seed + f); co = -np.ones(N, int)
        for k, c in enumerate(cl):
            co[c] = k
        mk_f = torch.tensor(build_masks(N, co, CFG.n_clients, CFG.seed + f)); Df, _ = make_features(trf)
        torch.manual_seed(CFG.seed + f)
        r_ = run_federated(Df, mk_f, trf, vaf, te, cl, CFG.cv_rounds, verbose=False)
        ev = evaluate_all(Df, mk_f, trf, te, r_['model'], r_['Dd'], r_['mk'])
        for nm, o in ev.items():
            for meth, P in o.items():
                CV[nm].setdefault(meth, []).append(100 * float((P.argmax(1) == Y[te]).mean()))
        print(f'fold {f + 1}/{CFG.n_folds}: multimodal Q-M2H acc {CV["multimodal"]["Proposed"][-1]:.1f}% | {time.time() - t0:.0f}s')
    rows = []
    for nm, o in CV.items():
        for meth, a in o.items():
            row = dict(modality=nm, method=meth, mean=np.mean(a), std=np.std(a, ddof=1)); row.update({f'fold{k + 1}': v for k, v in enumerate(a)})
            if meth != 'Proposed':
                tt, pp = stats.ttest_rel(o['Proposed'], a); row.update(t=tt, p=pp, significant=bool(pp < 0.05))
            rows.append(row)
    CVDF = pd.DataFrame(rows); CVDF.to_csv(f'{CFG.out_dir}/cv_results.csv', index=False)
    print(CVDF[['modality', 'method', 'mean', 'std', 't', 'p', 'significant']].round(4).to_string(index=False))
    fig, ax = plt.subplots(1, 6, figsize=(22, 3.6))
    for a_, nm in zip(ax, CFG.MODS):
        a_.boxplot([CV[nm][m] for m in CV[nm]], labels=list(CV[nm])); a_.set_title(f'{nm} (5-fold acc %)', fontsize=9); a_.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.savefig(f'{CFG.out_dir}/fig_cv.png', dpi=130); plt.show()

## 17. Robustness to missing modalities, ablations and the VDRC demo

In [ ]:
# (a) accuracy vs. forced extra missingness (every modality available -> dropped independently with probability r)
feats_main = modality_features(D); ef = make_xgb().fit(early_fusion_X(feats_main, MASK)[tr_idx], Y[tr_idx])
rr = np.random.default_rng(5); curve = []
for r_ in [0, .2, .4, .6, .8]:
    a_q, a_x = [], []
    for rep in range(5):
        mk = (rr.random((N, 6)) > r_).astype(np.float32); mk[mk.sum(1) == 0, 0] = 1; mk_t = torch.tensor(mk)
        a_q.append(100 * (predict(MODEL, Dd, mk_t.to(Dd['y'].device), te_idx).argmax(1) == y_te).mean())
        a_x.append(100 * (ef.predict_proba(early_fusion_X(feats_main, mk_t)[te_idx]).argmax(1) == y_te).mean())
    curve.append((r_, np.mean(a_q), np.mean(a_x)))
cdf = pd.DataFrame(curve, columns=['missing_rate', 'Q-M2H', 'XGB early-fusion']); print(cdf.round(2).to_string(index=False))
plt.figure(figsize=(5, 3.6)); plt.plot(cdf.missing_rate, cdf['Q-M2H'], 'o-', label='Q-M2H (MMA)'); plt.plot(cdf.missing_rate, cdf['XGB early-fusion'], 's--', label='XGB early fusion')
plt.xlabel('extra modality-missing rate'); plt.ylabel('test accuracy %'); plt.legend(); plt.title('Missing-modality robustness'); plt.tight_layout()
plt.savefig(f'{CFG.out_dir}/fig_missing_robustness.png', dpi=130); plt.show()

# (b) quantum noise mitigation (VDRC): error vs the ideal noiseless circuit, on circuit-0 of the trained model
with torch.no_grad():
    qc = MODEL.circ[0]; z_ = torch.randn(128, CFG.d, device=next(MODEL.parameters()).device)
    ideal, noisy, vd = qc(z_, 'ideal'), qc(z_, 'noisy'), qc(z_, 'vd')
print('VDRC  mean |error| vs ideal:  noisy %.4f -> virtual-distilled %.4f  (%.1fx lower)' % ((noisy - ideal).abs().mean(), (vd - ideal).abs().mean(), (noisy - ideal).abs().mean() / (vd - ideal).abs().mean()))

# (c) ablations (set CFG.run_ablation=True): each variant is retrained with the same federated protocol
if CFG.run_ablation:
    rows = []
    for name, kw in {'full Q-M2H': {}, 'no quantum layer': dict(use_quantum=False), 'no MMA': dict(use_mma=False), 'no entropy routing': dict(use_gate=False)}.items():
        torch.manual_seed(CFG.seed); r_ = run_federated(D, MASK, tr_idx, va_idx, te_idx, clients, CFG.cv_rounds, verbose=False, **kw)
        acc = lambda mk: 100 * (predict(r_['model'], r_['Dd'], mk, te_idx).argmax(1) == y_te).mean()
        mk50 = torch.tensor((np.random.default_rng(9).random((N, 6)) > 0.5).astype(np.float32)); mk50[mk50.sum(1) == 0, 0] = 1
        rows.append((name, acc(r_['mk']), acc(mk50.to(r_['mk'].device)), r_['n_params']))
    print(pd.DataFrame(rows, columns=['variant', 'acc (natural missing)', 'acc (50% extra missing)', 'params']).round(2).to_string(index=False))

## 18. Save artefacts

In [ ]:
torch.save(MODEL.state_dict(), f'{CFG.out_dir}/qm2h_global.pt')
json.dump({k: v for k, v in vars(CFG).items() if not k.startswith('_')}, open(f'{CFG.out_dir}/config.json', 'w'), indent=2, default=str)
pd.Series(full_m).to_csv(f'{CFG.out_dir}/main_metrics_3class.csv')
print('saved to', os.path.abspath(CFG.out_dir), '->', sorted(os.listdir(CFG.out_dir)))
try:
    from google.colab import files
    import shutil; shutil.make_archive('qm2h_outputs', 'zip', CFG.out_dir); files.download('qm2h_outputs.zip')
except Exception:
    pass